# Step 7 - Independent Validation

Executed project evidence and reproducible code.

# Step 7 — Final independent classical validation

In [64]:
from pathlib import Path
import importlib
import inspect
import json
import math
import re
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import files
STEP7_REQUIRED_OBJECTS = ['step4', 'step5', 'hybrid', 'STEP5_CONTEXT', 'STRICT_WARNING_CONTEXT', 'HYBRID_PREFERENCES', 'GOAL_SCALES', 'STEP5_PROFILE_RESULTS', 'STEP6_STEP7_HANDOFF', 'STEP6_SHORTLIST', 'HYBRID_RESULT', 'HYBRID_QAOA_PROFILE_RESULT', 'EXACT_ACTIVE_PROFILE_RESULT', 'QAOA_SEED_SUMMARY', 'DATA_SOURCE', 'STEP4_COST_SCENARIO', 'OUTPUT_ROOT', 'FAST_MODE']
STEP7_MISSING_OBJECTS = [name for name in STEP7_REQUIRED_OBJECTS if name not in globals()]
if STEP7_MISSING_OBJECTS:
    raise RuntimeError('Run the complete release Steps 3–6 workflow first. Missing Step 7 inputs: ' + ', '.join(STEP7_MISSING_OBJECTS))
if EXACT_ACTIVE_PROFILE_RESULT is None:
    raise RuntimeError('The exact active-set profile is required for Step 7.')
print('PASS: Step 7 runtime contract is complete.')
print('Step 6 shortlist:', list(STEP6_SHORTLIST.index))
print('QAOA seed count:', len(QAOA_SEED_SUMMARY))
print('Step 6 forward evidence ready:', STEP6_STEP7_HANDOFF['final_forward_evidence_ready'])


PASS: Step 7 runtime contract is complete.
Step 6 shortlist: ['Classical strict-warning reference', 'Primary unrestricted classical', 'Independent Qiskit QAOA']
QAOA seed count: 20
Step 6 forward evidence ready: True


## Install the independent Step 7 validation module

In [65]:
%%writefile step_07_validation_final.py
from __future__ import annotations
from dataclasses import dataclass
from itertools import combinations
from typing import Any, Iterable, Mapping, Sequence
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, OptimizeResult, minimize
EPS = 1e-12

def _array(values: Any) -> np.ndarray:
    if hasattr(values, 'to_numpy'):
        return values.to_numpy(dtype=float)
    return np.asarray(values, dtype=float)

@dataclass(frozen=True)
class ValidationTolerances:
    feasibility: float = 5e-06
    objective_match: float = 2e-05
    fixed_support_objective_match: float = 3e-05
    solver_weight_l1: float = 0.02
    qaoa_gap_zero: float = 1e-08
    hessian_psd: float = 1e-09
    kkt_stationarity: float = 0.0001
    kkt_complementarity: float = 1e-05
    dual_feasibility: float = 1e-07

@dataclass
class ClassicalValidationContext:
    step4: Any
    step5: Any
    hybrid: Any
    base_context: Any
    strict_context: Any | None
    preferences: Any
    scales: Any
    mix: Any
    profiles: Mapping[str, Mapping[str, Any]]
    step6_handoff: Mapping[str, Any]
    tolerances: ValidationTolerances = ValidationTolerances()

    def validate(self) -> None:
        self.base_context.portfolio_data.validate()
        self.base_context.scenarios.validate(len(self.base_context.portfolio_data.tickers))
        self.base_context.constraints.validate(self.base_context.portfolio_data)
        self.preferences.validate()
        self.scales.validate()
        self.mix.validate()
        if 'Primary unrestricted classical' not in self.profiles:
            raise ValueError('Primary classical profile is missing.')
        if 'Independent Qiskit QAOA' not in self.profiles:
            raise ValueError('Independent QAOA profile is missing.')
        if 'shortlist' not in self.step6_handoff:
            raise ValueError('The Step 6 shortlist is missing.')

@dataclass
class IndependentClassicalResult:
    label: str
    policy: str
    solver: str
    selected_solver_source: str
    success: bool
    message: str
    objective_value: float
    weights: pd.Series
    iterations: int
    optimality: float
    constr_violation: float
    kkt_stationarity_inf: float
    kkt_complementarity_inf: float
    dual_feasibility_min: float
    solver_reported_duality_gap: float
    kkt_certificate_pass: bool
    audit: pd.DataFrame
    objective_components: pd.Series
    solver_diagnostics: pd.DataFrame
    raw_result: Any
    fixed_active_tickers: tuple[str, ...] | None = None

@dataclass
class _ProblemDefinition:
    layout: dict[str, slice]
    lower_bounds: np.ndarray
    upper_bounds: np.ndarray
    constraint_matrix: np.ndarray
    constraint_lower: np.ndarray
    constraint_upper: np.ndarray
    objective_data: Any
    economic_data: Any
    objective_weights: Any
    coefficients: Mapping[str, float]
    context: Any
    fixed_active_tickers: tuple[str, ...] | None

def _layout(n_assets: int, n_scenarios: int) -> dict[str, slice]:
    cursor = 0
    result = {'w': slice(cursor, cursor + n_assets)}
    cursor += n_assets
    result['p'] = slice(cursor, cursor + n_assets)
    cursor += n_assets
    result['n'] = slice(cursor, cursor + n_assets)
    cursor += n_assets
    result['h'] = slice(cursor, cursor + n_scenarios)
    cursor += n_scenarios
    result['all'] = slice(0, cursor)
    return result

def _append(rows: list[np.ndarray], lower: list[float], upper: list[float], row: np.ndarray, lb: float, ub: float) -> None:
    rows.append(np.asarray(row, dtype=float))
    lower.append(float(lb))
    upper.append(float(ub))

def build_independent_problem(*, context: ClassicalValidationContext, policy_context: Any, fixed_active_tickers: Sequence[str] | None=None) -> _ProblemDefinition:
    economic_data = policy_context.portfolio_data
    objective_data, objective_weights, coefficients = context.step5.build_goal_objective(context.step4, context.preferences, economic_data, context.scales, context.mix)
    tickers = list(economic_data.tickers)
    n_assets = len(tickers)
    scenarios = policy_context.scenarios
    n_scenarios = len(scenarios.names)
    layout = _layout(n_assets, n_scenarios)
    n_variables = layout['all'].stop
    lower_bounds = np.full(n_variables, -np.inf, dtype=float)
    upper_bounds = np.full(n_variables, np.inf, dtype=float)
    lower_bounds[layout['w']] = _array(policy_context.constraints.asset_lower)
    upper_bounds[layout['w']] = _array(policy_context.constraints.asset_upper)
    lower_bounds[layout['p']] = 0.0
    lower_bounds[layout['n']] = 0.0
    lower_bounds[layout['h']] = 0.0
    upper_bounds[layout['p']] = 1.0
    upper_bounds[layout['n']] = 1.0
    rows: list[np.ndarray] = []
    lower: list[float] = []
    upper: list[float] = []
    current = _array(policy_context.current_weights)
    row = np.zeros(n_variables)
    row[layout['w']] = 1.0
    _append(rows, lower, upper, row, 1.0, 1.0)
    for index in range(n_assets):
        row = np.zeros(n_variables)
        row[layout['w'].start + index] = 1.0
        row[layout['p'].start + index] = -1.0
        row[layout['n'].start + index] = 1.0
        _append(rows, lower, upper, row, current[index], current[index])
    row = np.zeros(n_variables)
    row[layout['p']] = 1.0
    row[layout['n']] = 1.0
    _append(rows, lower, upper, row, -np.inf, policy_context.trading_config.turnover_limit_gross)
    trade_capacity = policy_context.trading_config.execution_days * policy_context.trading_config.participation_rate * _array(economic_data.adv_usd) / policy_context.trading_config.portfolio_value_usd
    for index in range(n_assets):
        row = np.zeros(n_variables)
        row[layout['p'].start + index] = 1.0
        row[layout['n'].start + index] = 1.0
        _append(rows, lower, upper, row, -np.inf, trade_capacity[index])
    asset_classes = np.asarray(economic_data.asset_classes, dtype=object)
    for class_name, (class_lower, class_upper) in policy_context.constraints.class_bounds.items():
        row = np.zeros(n_variables)
        row[layout['w']] = (asset_classes == class_name).astype(float)
        _append(rows, lower, upper, row, class_lower, class_upper)
    if economic_data.factor_loadings is not None and policy_context.constraints.factor_bounds:
        for factor, (factor_lower, factor_upper) in policy_context.constraints.factor_bounds.items():
            row = np.zeros(n_variables)
            row[layout['w']] = economic_data.factor_loadings[factor].reindex(tickers).to_numpy(dtype=float)
            _append(rows, lower, upper, row, factor_lower, factor_upper)
    if policy_context.constraints.income_floor is not None:
        row = np.zeros(n_variables)
        row[layout['w']] = _array(economic_data.income)
        _append(rows, lower, upper, row, policy_context.constraints.income_floor, np.inf)
    if policy_context.constraints.expected_total_return_floor is not None:
        row = np.zeros(n_variables)
        row[layout['w']] = _array(economic_data.total_return)
        _append(rows, lower, upper, row, policy_context.constraints.expected_total_return_floor, np.inf)
    loss_matrix = _array(scenarios.loss_matrix)
    for scenario_index in range(n_scenarios):
        row = np.zeros(n_variables)
        row[layout['w']] = loss_matrix[scenario_index]
        row[layout['h'].start + scenario_index] = -1.0
        _append(rows, lower, upper, row, -np.inf, scenarios.warning_thresholds[scenario_index])
        row = np.zeros(n_variables)
        row[layout['w']] = loss_matrix[scenario_index]
        _append(rows, lower, upper, row, -np.inf, scenarios.hard_loss_limits[scenario_index])
    fixed_tuple: tuple[str, ...] | None = None
    if fixed_active_tickers is not None:
        fixed_tuple = tuple((str(value) for value in fixed_active_tickers))
        unknown = sorted(set(fixed_tuple) - set(tickers))
        if unknown:
            raise ValueError('Unknown fixed-support tickers: ' + ', '.join(unknown))
        active = set(fixed_tuple)
        for index, ticker in enumerate(tickers):
            if ticker in active:
                continue
            row = np.zeros(n_variables)
            row[layout['w'].start + index] = 1.0
            _append(rows, lower, upper, row, current[index], current[index])
    return _ProblemDefinition(layout=layout, lower_bounds=lower_bounds, upper_bounds=upper_bounds, constraint_matrix=np.vstack(rows), constraint_lower=np.asarray(lower, dtype=float), constraint_upper=np.asarray(upper, dtype=float), objective_data=objective_data, economic_data=economic_data, objective_weights=objective_weights, coefficients=coefficients, context=policy_context, fixed_active_tickers=fixed_tuple)

def _objective_functions(problem: _ProblemDefinition):
    layout = problem.layout
    data = problem.objective_data
    context = problem.context
    scenarios = context.scenarios
    current = _array(context.current_weights)
    covariance = _array(data.covariance)
    impact = _array(data.impact_matrix)
    ow = problem.objective_weights

    def objective(x: np.ndarray) -> float:
        w = x[layout['w']]
        p = x[layout['p']]
        n = x[layout['n']]
        h = x[layout['h']]
        delta = w - current
        return float(ow.risk_penalty * (w @ covariance @ w) - ow.growth_reward * (_array(data.growth) @ w) - ow.income_reward * (_array(data.income) @ w) + ow.concentration_penalty * (w @ w) + ow.transaction_cost_penalty * (_array(data.linear_cost) @ (p + n)) + ow.market_impact_penalty * (delta @ impact @ delta) + ow.scenario_penalty * (_array(scenarios.weights) @ (h * h)))

    def gradient(x: np.ndarray) -> np.ndarray:
        w = x[layout['w']]
        h = x[layout['h']]
        delta = w - current
        grad = np.zeros_like(x)
        grad[layout['w']] = 2.0 * ow.risk_penalty * covariance @ w - ow.growth_reward * _array(data.growth) - ow.income_reward * _array(data.income) + 2.0 * ow.concentration_penalty * w + 2.0 * ow.market_impact_penalty * impact @ delta
        grad[layout['p']] = ow.transaction_cost_penalty * _array(data.linear_cost)
        grad[layout['n']] = ow.transaction_cost_penalty * _array(data.linear_cost)
        grad[layout['h']] = 2.0 * ow.scenario_penalty * _array(scenarios.weights) * h
        return grad
    n_variables = layout['all'].stop
    hessian_matrix = np.zeros((n_variables, n_variables), dtype=float)
    hessian_matrix[layout['w'], layout['w']] = 2.0 * ow.risk_penalty * covariance + 2.0 * ow.concentration_penalty * np.eye(len(data.tickers)) + 2.0 * ow.market_impact_penalty * impact
    hessian_matrix[layout['h'], layout['h']] = np.diag(2.0 * ow.scenario_penalty * _array(scenarios.weights))

    def hessian(_: np.ndarray) -> np.ndarray:
        return hessian_matrix
    return (objective, gradient, hessian, hessian_matrix)

def _initial_point(problem: _ProblemDefinition, weights: Any) -> np.ndarray:
    layout = problem.layout
    context = problem.context
    scenarios = context.scenarios
    current = _array(context.current_weights)
    w = _array(weights).copy()
    if w.shape != current.shape:
        raise ValueError('Starting weights have the wrong shape.')
    x = np.zeros(layout['all'].stop, dtype=float)
    x[layout['w']] = w
    delta = w - current
    x[layout['p']] = np.maximum(delta, 0.0)
    x[layout['n']] = np.maximum(-delta, 0.0)
    losses = _array(scenarios.loss_matrix) @ w
    x[layout['h']] = np.maximum(losses - _array(scenarios.warning_thresholds), 0.0)
    return x

def objective_components_from_weights(*, validation_context: ClassicalValidationContext, policy_context: Any, weights: Any) -> pd.Series:
    objective_data, objective_weights, _ = validation_context.step5.build_goal_objective(validation_context.step4, validation_context.preferences, policy_context.portfolio_data, validation_context.scales, validation_context.mix)
    w = _array(weights)
    current = _array(policy_context.current_weights)
    delta = w - current
    losses = _array(policy_context.scenarios.loss_matrix) @ w
    h = np.maximum(losses - _array(policy_context.scenarios.warning_thresholds), 0.0)
    p_plus_n = np.abs(delta)
    components = {'growth_reward': -float(objective_weights.growth_reward * (_array(objective_data.growth) @ w)), 'income_reward': -float(objective_weights.income_reward * (_array(objective_data.income) @ w)), 'variance_penalty': float(objective_weights.risk_penalty * (w @ _array(objective_data.covariance) @ w)), 'concentration_penalty': float(objective_weights.concentration_penalty * (w @ w)), 'linear_and_turnover_cost_penalty': float(objective_weights.transaction_cost_penalty * (_array(objective_data.linear_cost) @ p_plus_n)), 'market_impact_penalty': float(objective_weights.market_impact_penalty * (delta @ _array(objective_data.impact_matrix) @ delta)), 'scenario_hinge_penalty': float(objective_weights.scenario_penalty * (_array(policy_context.scenarios.weights) @ (h * h)))}
    components['total_objective'] = float(sum(components.values()))
    return pd.Series(components, dtype=float)

def independent_constraint_audit(*, policy_context: Any, weights: Any, fixed_active_tickers: Sequence[str] | None=None, tolerance: float=5e-06) -> pd.DataFrame:
    data = policy_context.portfolio_data
    constraints = policy_context.constraints
    trading = policy_context.trading_config
    scenarios = policy_context.scenarios
    tickers = list(data.tickers)
    w = _array(weights)
    current = _array(policy_context.current_weights)
    delta = w - current
    abs_trade = np.abs(delta)
    records: list[dict[str, Any]] = []

    def add(category: str, name: str, value: float, lower: float, upper: float) -> None:
        lower_violation = max(lower - value, 0.0) if np.isfinite(lower) else 0.0
        upper_violation = max(value - upper, 0.0) if np.isfinite(upper) else 0.0
        violation = max(lower_violation, upper_violation)
        records.append({'category': category, 'constraint': name, 'value': float(value), 'lower': float(lower), 'upper': float(upper), 'absolute_violation': float(violation), 'satisfied': bool(violation <= tolerance)})
    add('budget', 'sum_weights', float(w.sum()), 1.0, 1.0)
    for index, ticker in enumerate(tickers):
        add('asset', f'weight_{ticker}', w[index], constraints.asset_lower[index], constraints.asset_upper[index])
    asset_classes = np.asarray(data.asset_classes, dtype=object)
    for class_name, (class_lower, class_upper) in constraints.class_bounds.items():
        exposure = float(w[asset_classes == class_name].sum())
        add('asset_class', f'class_{class_name}', exposure, class_lower, class_upper)
    if data.factor_loadings is not None:
        for factor, (factor_lower, factor_upper) in constraints.factor_bounds.items():
            exposure = float(data.factor_loadings[factor].reindex(tickers).to_numpy(dtype=float) @ w)
            add('factor', f'factor_{factor}', exposure, factor_lower, factor_upper)
    if constraints.income_floor is not None:
        add('income', 'income_floor', float(_array(data.income) @ w), constraints.income_floor, np.inf)
    if constraints.expected_total_return_floor is not None:
        add('return', 'expected_total_return_floor', float(_array(data.total_return) @ w), constraints.expected_total_return_floor, np.inf)
    add('turnover', 'gross_turnover', float(abs_trade.sum()), -np.inf, trading.turnover_limit_gross)
    trade_capacity = trading.execution_days * trading.participation_rate * _array(data.adv_usd) / trading.portfolio_value_usd
    for index, ticker in enumerate(tickers):
        add('liquidity', f'trade_capacity_{ticker}', abs_trade[index], -np.inf, trade_capacity[index])
    losses = _array(scenarios.loss_matrix) @ w
    for scenario_index, scenario_name in enumerate(scenarios.names):
        add('scenario', f'scenario_{scenario_name}', losses[scenario_index], -np.inf, scenarios.hard_loss_limits[scenario_index])
    if fixed_active_tickers is not None:
        active = set((str(value) for value in fixed_active_tickers))
        for index, ticker in enumerate(tickers):
            if ticker not in active:
                add('fixed_support', f'inactive_trade_{ticker}', delta[index], 0.0, 0.0)
    return pd.DataFrame(records)

def cvxpy_solver_availability() -> pd.Series:
    try:
        import cvxpy as cp
    except Exception as exc:
        return pd.Series({'cvxpy_available': False, 'cvxpy_version': np.nan, 'installed_solvers': '', 'clarabel_available': False, 'osqp_available': False, 'error': f'{type(exc).__name__}: {exc}'}, name='value')
    installed = tuple(sorted((str(value) for value in cp.installed_solvers())))
    return pd.Series({'cvxpy_available': True, 'cvxpy_version': str(cp.__version__), 'installed_solvers': ', '.join(installed), 'clarabel_available': 'CLARABEL' in installed, 'osqp_available': 'OSQP' in installed, 'error': ''}, name='value')

def _constraint_partitions(problem: _ProblemDefinition, *, equality_tolerance: float=1e-12) -> dict[str, np.ndarray]:
    lower = problem.constraint_lower
    upper = problem.constraint_upper
    finite_lower = np.isfinite(lower)
    finite_upper = np.isfinite(upper)
    equality = finite_lower & finite_upper & (np.abs(lower - upper) <= equality_tolerance)
    return {'equality': equality, 'lower_inequality': finite_lower & ~equality, 'upper_inequality': finite_upper & ~equality, 'non_equality': ~equality}

def _split_scipy_linear_constraints(problem: _ProblemDefinition) -> list[LinearConstraint]:
    partitions = _constraint_partitions(problem)
    constraints: list[LinearConstraint] = []
    equality = partitions['equality']
    if equality.any():
        equality_rhs = problem.constraint_lower[equality]
        constraints.append(LinearConstraint(problem.constraint_matrix[equality], equality_rhs, equality_rhs))
    non_equality = partitions['non_equality']
    if non_equality.any():
        constraints.append(LinearConstraint(problem.constraint_matrix[non_equality], problem.constraint_lower[non_equality], problem.constraint_upper[non_equality]))
    return constraints

def _full_primal_residual(problem: _ProblemDefinition, x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    if x.shape != problem.lower_bounds.shape or not np.isfinite(x).all():
        return float('inf')
    bound_lower = np.where(np.isfinite(problem.lower_bounds), np.maximum(problem.lower_bounds - x, 0.0), 0.0)
    bound_upper = np.where(np.isfinite(problem.upper_bounds), np.maximum(x - problem.upper_bounds, 0.0), 0.0)
    values = problem.constraint_matrix @ x
    row_lower = np.where(np.isfinite(problem.constraint_lower), np.maximum(problem.constraint_lower - values, 0.0), 0.0)
    row_upper = np.where(np.isfinite(problem.constraint_upper), np.maximum(values - problem.constraint_upper, 0.0), 0.0)
    return float(max(bound_lower.max(initial=0.0), bound_upper.max(initial=0.0), row_lower.max(initial=0.0), row_upper.max(initial=0.0)))

def _extract_reported_gap(extra_stats: Any) -> float:
    if extra_stats is None:
        return float('nan')
    candidates = ('gap_abs', 'duality_gap', 'dual_gap', 'gap', 'rel_gap')

    def lookup(value: Any) -> float | None:
        if value is None:
            return None
        if isinstance(value, Mapping):
            for key in candidates:
                if key in value:
                    try:
                        return float(value[key])
                    except Exception:
                        pass
            for nested in value.values():
                result = lookup(nested)
                if result is not None:
                    return result
            return None
        for key in candidates:
            if hasattr(value, key):
                try:
                    return float(getattr(value, key))
                except Exception:
                    pass
        for nested_name in ('info', 'solution', 'stats'):
            if hasattr(value, nested_name):
                result = lookup(getattr(value, nested_name))
                if result is not None:
                    return result
        return None
    result = lookup(extra_stats)
    return float('nan') if result is None else float(result)

def _cvxpy_kkt_diagnostics(*, problem: _ProblemDefinition, x_value: np.ndarray, hessian_matrix: np.ndarray, linear_term: np.ndarray, dual_objects: Mapping[str, Any]) -> dict[str, float | bool]:
    x_value = np.asarray(x_value, dtype=float)
    stationarity = hessian_matrix @ x_value + linear_term
    complementarity_terms: list[np.ndarray] = []
    inequality_duals: list[np.ndarray] = []

    def dual_array(name: str) -> np.ndarray | None:
        constraint = dual_objects.get(name)
        if constraint is None or constraint.dual_value is None:
            return None
        return np.asarray(constraint.dual_value, dtype=float).reshape(-1)
    partitions = _constraint_partitions(problem)
    matrix = problem.constraint_matrix
    equality = partitions['equality']
    equality_dual = dual_array('row_equality')
    if equality.any() and equality_dual is not None:
        stationarity = stationarity + matrix[equality].T @ equality_dual
    lower_mask = partitions['lower_inequality']
    lower_dual = dual_array('row_lower')
    if lower_mask.any() and lower_dual is not None:
        stationarity = stationarity - matrix[lower_mask].T @ lower_dual
        lower_slack = matrix[lower_mask] @ x_value - problem.constraint_lower[lower_mask]
        complementarity_terms.append(lower_dual * lower_slack)
        inequality_duals.append(lower_dual)
    upper_mask = partitions['upper_inequality']
    upper_dual = dual_array('row_upper')
    if upper_mask.any() and upper_dual is not None:
        stationarity = stationarity + matrix[upper_mask].T @ upper_dual
        upper_slack = problem.constraint_upper[upper_mask] - matrix[upper_mask] @ x_value
        complementarity_terms.append(upper_dual * upper_slack)
        inequality_duals.append(upper_dual)
    finite_bound_lower = np.isfinite(problem.lower_bounds)
    bound_lower_dual = dual_array('bound_lower')
    if finite_bound_lower.any() and bound_lower_dual is not None:
        stationarity[finite_bound_lower] -= bound_lower_dual
        lower_slack = x_value[finite_bound_lower] - problem.lower_bounds[finite_bound_lower]
        complementarity_terms.append(bound_lower_dual * lower_slack)
        inequality_duals.append(bound_lower_dual)
    finite_bound_upper = np.isfinite(problem.upper_bounds)
    bound_upper_dual = dual_array('bound_upper')
    if finite_bound_upper.any() and bound_upper_dual is not None:
        stationarity[finite_bound_upper] += bound_upper_dual
        upper_slack = problem.upper_bounds[finite_bound_upper] - x_value[finite_bound_upper]
        complementarity_terms.append(bound_upper_dual * upper_slack)
        inequality_duals.append(bound_upper_dual)
    complementarity = max((float(np.abs(values).max(initial=0.0)) for values in complementarity_terms)) if complementarity_terms else 0.0
    dual_minimum = min((float(values.min(initial=0.0)) for values in inequality_duals)) if inequality_duals else 0.0
    return {'primal_residual_inf': _full_primal_residual(problem, x_value), 'stationarity_residual_inf': float(np.abs(stationarity).max(initial=0.0)), 'complementarity_residual_inf': float(complementarity), 'dual_feasibility_min': float(dual_minimum)}

def _solve_with_cvxpy(*, problem: _ProblemDefinition, x0: np.ndarray, tolerances: ValidationTolerances) -> list[dict[str, Any]]:
    try:
        import cvxpy as cp
    except Exception as exc:
        raise RuntimeError('CVXPY is required for the final independent Step 7 validation. Install cvxpy, clarabel, and osqp.') from exc
    installed = set((str(value) for value in cp.installed_solvers()))
    solver_order = [solver for solver in ('CLARABEL', 'OSQP') if solver in installed]
    if not solver_order:
        raise RuntimeError('Neither CLARABEL nor OSQP is available through CVXPY.')
    _, gradient, _, hessian_matrix = _objective_functions(problem)
    hessian_matrix = 0.5 * (hessian_matrix + hessian_matrix.T)
    linear_term = gradient(np.zeros(problem.layout['all'].stop, dtype=float))
    n_variables = problem.layout['all'].stop
    variable = cp.Variable(n_variables, name='portfolio_qp_variables')
    dual_objects: dict[str, Any] = {}
    constraints: list[Any] = []
    finite_lower = np.isfinite(problem.lower_bounds)
    if finite_lower.any():
        dual_objects['bound_lower'] = variable[finite_lower] >= problem.lower_bounds[finite_lower]
        constraints.append(dual_objects['bound_lower'])
    finite_upper = np.isfinite(problem.upper_bounds)
    if finite_upper.any():
        dual_objects['bound_upper'] = variable[finite_upper] <= problem.upper_bounds[finite_upper]
        constraints.append(dual_objects['bound_upper'])
    partitions = _constraint_partitions(problem)
    matrix = problem.constraint_matrix
    equality = partitions['equality']
    if equality.any():
        dual_objects['row_equality'] = matrix[equality] @ variable == problem.constraint_lower[equality]
        constraints.append(dual_objects['row_equality'])
    lower_mask = partitions['lower_inequality']
    if lower_mask.any():
        dual_objects['row_lower'] = matrix[lower_mask] @ variable >= problem.constraint_lower[lower_mask]
        constraints.append(dual_objects['row_lower'])
    upper_mask = partitions['upper_inequality']
    if upper_mask.any():
        dual_objects['row_upper'] = matrix[upper_mask] @ variable <= problem.constraint_upper[upper_mask]
        constraints.append(dual_objects['row_upper'])
    qp_objective = cp.Minimize(0.5 * cp.quad_form(variable, cp.psd_wrap(hessian_matrix)) + linear_term @ variable)
    cvx_problem = cp.Problem(qp_objective, constraints)
    records: list[dict[str, Any]] = []
    for solver_name in solver_order:
        variable.value = np.asarray(x0, dtype=float)
        solver_options: dict[str, Any]
        if solver_name == 'CLARABEL':
            solver_options = {'max_iter': 5000, 'tol_gap_abs': 1e-10, 'tol_gap_rel': 1e-10, 'tol_feas': 1e-10}
        else:
            solver_options = {'max_iter': 200000, 'eps_abs': 1e-09, 'eps_rel': 1e-09, 'polishing': True}
        try:
            cvx_problem.solve(solver=solver_name, warm_start=True, verbose=False, **solver_options)
            status = str(cvx_problem.status)
            x_value = None if variable.value is None else np.asarray(variable.value, dtype=float).copy()
            finite = bool(x_value is not None and x_value.shape == (n_variables,) and np.isfinite(x_value).all())
            if finite:
                kkt = _cvxpy_kkt_diagnostics(problem=problem, x_value=x_value, hessian_matrix=hessian_matrix, linear_term=linear_term, dual_objects=dual_objects)
            else:
                kkt = {'primal_residual_inf': float('inf'), 'stationarity_residual_inf': float('inf'), 'complementarity_residual_inf': float('inf'), 'dual_feasibility_min': float('-inf')}
            reported_gap = _extract_reported_gap(getattr(cvx_problem.solver_stats, 'extra_stats', None))
            kkt_pass = bool(finite and status in {str(cp.OPTIMAL), str(cp.OPTIMAL_INACCURATE)} and (kkt['primal_residual_inf'] <= tolerances.feasibility) and (kkt['stationarity_residual_inf'] <= tolerances.kkt_stationarity) and (kkt['complementarity_residual_inf'] <= tolerances.kkt_complementarity) and (kkt['dual_feasibility_min'] >= -tolerances.dual_feasibility))
            records.append({'solver_source': f'cvxpy_{solver_name.lower()}', 'role': 'independent_convex_qp_validator', 'status': status, 'finite_solution': finite, 'x': x_value, 'primal_residual_inf': float(kkt['primal_residual_inf']), 'stationarity_residual_inf': float(kkt['stationarity_residual_inf']), 'complementarity_residual_inf': float(kkt['complementarity_residual_inf']), 'dual_feasibility_min': float(kkt['dual_feasibility_min']), 'solver_reported_duality_gap': reported_gap, 'kkt_certificate_pass': kkt_pass, 'iterations': int(getattr(cvx_problem.solver_stats, 'num_iters', 0) or 0), 'solve_time_seconds': float(getattr(cvx_problem.solver_stats, 'solve_time', np.nan) or np.nan), 'raw_result': {'status': status, 'solver_name': solver_name, 'problem_value_without_constant': None if cvx_problem.value is None else float(cvx_problem.value), 'num_iters': int(getattr(cvx_problem.solver_stats, 'num_iters', 0) or 0), 'solve_time_seconds': float(getattr(cvx_problem.solver_stats, 'solve_time', np.nan) or np.nan), 'setup_time_seconds': float(getattr(cvx_problem.solver_stats, 'setup_time', np.nan) or np.nan), 'extra_stats_type': type(getattr(cvx_problem.solver_stats, 'extra_stats', None)).__name__}, 'error': ''})
        except Exception as exc:
            records.append({'solver_source': f'cvxpy_{solver_name.lower()}', 'role': 'independent_convex_qp_validator', 'status': 'ERROR', 'finite_solution': False, 'x': None, 'primal_residual_inf': float('inf'), 'stationarity_residual_inf': float('inf'), 'complementarity_residual_inf': float('inf'), 'dual_feasibility_min': float('-inf'), 'solver_reported_duality_gap': float('nan'), 'kkt_certificate_pass': False, 'iterations': 0, 'solve_time_seconds': float('nan'), 'raw_result': None, 'error': f'{type(exc).__name__}: {exc}'})
    return records

def _scipy_diagnostic_record(*, solver_source: str, role: str, result: OptimizeResult, problem: _ProblemDefinition, validation_context: ClassicalValidationContext, policy_context: Any, fixed_active_tickers: Sequence[str] | None) -> dict[str, Any]:
    x_value = np.asarray(getattr(result, 'x', np.full(problem.layout['all'].stop, np.nan)), dtype=float)
    finite = bool(x_value.shape == (problem.layout['all'].stop,) and np.isfinite(x_value).all())
    if finite:
        weights = x_value[problem.layout['w']]
        audit = independent_constraint_audit(policy_context=policy_context, weights=weights, fixed_active_tickers=fixed_active_tickers, tolerance=validation_context.tolerances.feasibility)
        objective_value = float(objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=weights)['total_objective'])
        feasible = bool(audit['satisfied'].all() and _full_primal_residual(problem, x_value) <= validation_context.tolerances.feasibility)
    else:
        objective_value = float('nan')
        feasible = False
    return {'solver_source': solver_source, 'role': role, 'status': str(getattr(result, 'message', '')), 'finite_solution': finite, 'feasible': feasible, 'exact_economic_objective': objective_value, 'primal_residual_inf': _full_primal_residual(problem, x_value) if finite else float('inf'), 'stationarity_residual_inf': float(getattr(result, 'optimality', np.nan)), 'complementarity_residual_inf': float('nan'), 'dual_feasibility_min': float('nan'), 'solver_reported_duality_gap': float('nan'), 'kkt_certificate_pass': False, 'iterations': int(getattr(result, 'nit', 0) or 0), 'solve_time_seconds': float('nan'), 'selected': False, 'error': ''}

def solve_independent_classical(*, validation_context: ClassicalValidationContext, policy_context: Any, label: str, policy: str, start_weights: Any, fixed_active_tickers: Sequence[str] | None=None, maxiter: int=2000) -> IndependentClassicalResult:
    problem = build_independent_problem(context=validation_context, policy_context=policy_context, fixed_active_tickers=fixed_active_tickers)
    objective, gradient, hessian, _ = _objective_functions(problem)
    x0 = _initial_point(problem, start_weights)
    scipy_constraints = _split_scipy_linear_constraints(problem)
    bounds = Bounds(problem.lower_bounds, problem.upper_bounds)
    trust_result = minimize(objective, x0, method='trust-constr', jac=gradient, hess=hessian, bounds=bounds, constraints=scipy_constraints, options={'gtol': 1e-09, 'xtol': 1e-11, 'barrier_tol': 1e-11, 'maxiter': int(maxiter), 'verbose': 0})
    trust_x = np.asarray(getattr(trust_result, 'x', x0), dtype=float)
    polish_start = trust_x if trust_x.shape == x0.shape and np.isfinite(trust_x).all() else x0.copy()
    polish_result = minimize(objective, polish_start, method='SLSQP', jac=gradient, bounds=bounds, constraints=scipy_constraints, options={'ftol': 1e-12, 'maxiter': max(3000, int(maxiter)), 'disp': False})
    scipy_records = [_scipy_diagnostic_record(solver_source='scipy_trust_constr', role='algorithmic_cross_check', result=trust_result, problem=problem, validation_context=validation_context, policy_context=policy_context, fixed_active_tickers=fixed_active_tickers), _scipy_diagnostic_record(solver_source='scipy_slsqp_split_constraints', role='numerical_cross_check_only', result=polish_result, problem=problem, validation_context=validation_context, policy_context=policy_context, fixed_active_tickers=fixed_active_tickers)]
    cvxpy_records = _solve_with_cvxpy(problem=problem, x0=x0, tolerances=validation_context.tolerances)
    eligible_cvxpy_records: list[dict[str, Any]] = []
    for record in cvxpy_records:
        candidate = record.get('x')
        if candidate is None:
            record['feasible'] = False
            record['exact_economic_objective'] = float('nan')
            record['selected'] = False
            continue
        candidate = np.asarray(candidate, dtype=float)
        weights = candidate[problem.layout['w']]
        audit = independent_constraint_audit(policy_context=policy_context, weights=weights, fixed_active_tickers=fixed_active_tickers, tolerance=validation_context.tolerances.feasibility)
        exact_objective = float(objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=weights)['total_objective'])
        feasible = bool(audit['satisfied'].all() and record['primal_residual_inf'] <= validation_context.tolerances.feasibility)
        record['feasible'] = feasible
        record['exact_economic_objective'] = exact_objective
        record['selected'] = False
        if feasible and bool(record['kkt_certificate_pass']) and np.isfinite(exact_objective):
            record['_audit'] = audit
            eligible_cvxpy_records.append(record)
    if not eligible_cvxpy_records:
        diagnostic_table = pd.DataFrame([{key: value for key, value in record.items() if key not in {'x', 'raw_result', '_audit'}} for record in cvxpy_records])
        raise RuntimeError('No independent CVXPY solver produced a feasible KKT-certified Step 7 solution. Diagnostics:\n' + diagnostic_table.to_string(index=False))
    selected = min(eligible_cvxpy_records, key=lambda record: record['exact_economic_objective'])
    selected['selected'] = True
    final_x = np.asarray(selected['x'], dtype=float)
    final_weights = final_x[problem.layout['w']]
    final_audit = selected['_audit']
    all_records = scipy_records + cvxpy_records
    diagnostics_rows = []
    for record in all_records:
        diagnostics_rows.append({'solver_source': record['solver_source'], 'role': record['role'], 'status': record['status'], 'finite_solution': bool(record.get('finite_solution', False)), 'feasible': bool(record.get('feasible', False)), 'exact_economic_objective': float(record.get('exact_economic_objective', np.nan)), 'primal_residual_inf': float(record.get('primal_residual_inf', np.nan)), 'stationarity_residual_inf': float(record.get('stationarity_residual_inf', np.nan)), 'complementarity_residual_inf': float(record.get('complementarity_residual_inf', np.nan)), 'dual_feasibility_min': float(record.get('dual_feasibility_min', np.nan)), 'solver_reported_duality_gap': float(record.get('solver_reported_duality_gap', np.nan)), 'kkt_certificate_pass': bool(record.get('kkt_certificate_pass', False)), 'iterations': int(record.get('iterations', 0)), 'solve_time_seconds': float(record.get('solve_time_seconds', np.nan)), 'selected': bool(record.get('selected', False)), 'error': str(record.get('error', ''))})
    solver_diagnostics = pd.DataFrame(diagnostics_rows)
    components = objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=final_weights)
    feasible = bool(final_audit['satisfied'].all())
    kkt_pass = bool(selected['kkt_certificate_pass'])
    combined_message = '; '.join([f"selected independent solver={selected['solver_source']}", f'trust-constr={trust_result.message}', f'split-constraint SLSQP={polish_result.message}', 'stored starting point was initialization only and was not selectable'])
    return IndependentClassicalResult(label=label, policy=policy, solver='independent CVXPY convex QP (Clarabel/OSQP)', selected_solver_source=str(selected['solver_source']), success=bool(feasible and kkt_pass and np.isfinite(components['total_objective'])), message=combined_message, objective_value=float(components['total_objective']), weights=pd.Series(final_weights, index=policy_context.portfolio_data.tickers, name=label), iterations=int(selected['iterations']), optimality=float(selected['stationarity_residual_inf']), constr_violation=float(selected['primal_residual_inf']), kkt_stationarity_inf=float(selected['stationarity_residual_inf']), kkt_complementarity_inf=float(selected['complementarity_residual_inf']), dual_feasibility_min=float(selected['dual_feasibility_min']), solver_reported_duality_gap=float(selected['solver_reported_duality_gap']), kkt_certificate_pass=kkt_pass, audit=final_audit, objective_components=components, solver_diagnostics=solver_diagnostics, raw_result=selected['raw_result'], fixed_active_tickers=None if fixed_active_tickers is None else tuple((str(value) for value in fixed_active_tickers)))

def convexity_certificate(*, validation_context: ClassicalValidationContext, policy_context: Any) -> pd.Series:
    problem = build_independent_problem(context=validation_context, policy_context=policy_context)
    _, _, _, hessian = _objective_functions(problem)
    covariance_min = float(np.linalg.eigvalsh(_array(problem.objective_data.covariance)).min())
    impact_min = float(np.linalg.eigvalsh(_array(problem.objective_data.impact_matrix)).min())
    hessian_min = float(np.linalg.eigvalsh(hessian).min())
    coefficients_nonnegative = all((float(value) >= -validation_context.tolerances.hessian_psd for key, value in problem.coefficients.items() if key not in {'growth_coefficient', 'income_coefficient'}))
    certified = bool(covariance_min >= -validation_context.tolerances.hessian_psd and impact_min >= -validation_context.tolerances.hessian_psd and (hessian_min >= -validation_context.tolerances.hessian_psd) and coefficients_nonnegative)
    return pd.Series({'covariance_minimum_eigenvalue': covariance_min, 'impact_minimum_eigenvalue': impact_min, 'objective_hessian_minimum_eigenvalue': hessian_min, 'penalty_coefficients_nonnegative': coefficients_nonnegative, 'linear_constraint_system': True, 'continuous_problem_convex': certified, 'global_optimum_interpretation': certified}, name='value')

def compare_profile_to_solution(*, validation_context: ClassicalValidationContext, policy_context: Any, profile_name: str, profile: Mapping[str, Any], independent_result: IndependentClassicalResult, validation_target: str, objective_tolerance: float | None=None) -> pd.Series:
    stored_weights = _array(profile['result'].weights)
    stored_components = objective_components_from_weights(validation_context=validation_context, policy_context=policy_context, weights=stored_weights)
    stored_audit = independent_constraint_audit(policy_context=policy_context, weights=stored_weights, fixed_active_tickers=independent_result.fixed_active_tickers, tolerance=validation_context.tolerances.feasibility)
    independent_weights = independent_result.weights.to_numpy(dtype=float)
    raw_gap = float(stored_components['total_objective'] - independent_result.objective_value)
    numerical_gap = max(raw_gap, 0.0)
    tolerance = validation_context.tolerances.objective_match if objective_tolerance is None else float(objective_tolerance)
    feasible = bool(stored_audit['satisfied'].all())
    objective_match = bool(abs(raw_gap) <= tolerance)
    if not independent_result.success:
        verdict = 'FAIL_INDEPENDENT_SOLVER'
    elif not feasible:
        verdict = 'FAIL_INFEASIBLE'
    elif objective_match:
        verdict = 'PASS_OPTIMUM_MATCH'
    else:
        verdict = 'PASS_FEASIBLE_WITH_GAP'
    economic = validation_context.step5.exact_goal_components(policy_context.portfolio_data, stored_weights, policy_context.current_weights, policy_context.scenarios)
    return pd.Series({'profile': profile_name, 'validation_target': validation_target, 'policy': independent_result.policy, 'stored_objective': float(stored_components['total_objective']), 'independent_objective': independent_result.objective_value, 'signed_objective_gap': raw_gap, 'nonnegative_objective_gap': numerical_gap, 'objective_match_tolerance': tolerance, 'weight_l1_difference': float(np.abs(stored_weights - independent_weights).sum()), 'weight_linf_difference': float(np.abs(stored_weights - independent_weights).max()), 'expected_total_return': economic['expected_total_return'], 'volatility': economic['volatility'], 'worst_scenario_loss': economic['worst_scenario_loss'], 'gross_turnover': economic['gross_turnover'], 'total_trading_cost': economic['total_trading_cost'], 'hard_constraint_pass': feasible, 'independent_solver_success': independent_result.success, 'selected_solver_source': independent_result.selected_solver_source, 'kkt_certificate_pass': independent_result.kkt_certificate_pass, 'kkt_stationarity_inf': independent_result.kkt_stationarity_inf, 'kkt_complementarity_inf': independent_result.kkt_complementarity_inf, 'dual_feasibility_min': independent_result.dual_feasibility_min, 'verdict': verdict}, name=profile_name)

def enumerate_qubo_exactly(*, model: Any, qaoa_selected_tickers: Sequence[str], exact_selected_tickers: Sequence[str] | None=None) -> tuple[pd.DataFrame, pd.Series]:
    n = int(model.n_variables)
    cardinality = int(model.cardinality)
    records: list[dict[str, Any]] = []
    for chosen in combinations(range(n), cardinality):
        bits = np.zeros(n, dtype=int)
        bits[list(chosen)] = 1
        selected = tuple((model.tickers[index] for index in chosen))
        records.append({'bitstring': ''.join((str(int(value)) for value in bits)), 'energy': float(model.energy(bits)), 'selected_tickers': selected})
    table = pd.DataFrame(records).sort_values(['energy', 'bitstring']).reset_index(drop=True)
    table['classical_rank'] = np.arange(1, len(table) + 1)
    qaoa_set = frozenset((str(value) for value in qaoa_selected_tickers))
    qaoa_matches = table['selected_tickers'].map(lambda values: frozenset(values) == qaoa_set)
    if not qaoa_matches.any():
        raise ValueError('QAOA selected set is absent from exact enumeration.')
    qaoa_row = table.loc[qaoa_matches].iloc[0]
    exact_row = table.iloc[0]
    exact_set_matches = np.nan
    if exact_selected_tickers is not None:
        exact_set_matches = bool(frozenset(exact_selected_tickers) == frozenset(exact_row['selected_tickers']))
    summary = pd.Series({'state_count': len(table), 'n_variables': n, 'cardinality': cardinality, 'exact_energy': float(exact_row['energy']), 'qaoa_energy': float(qaoa_row['energy']), 'qaoa_energy_gap': float(qaoa_row['energy'] - exact_row['energy']), 'qaoa_classical_rank': int(qaoa_row['classical_rank']), 'qaoa_exact_optimum': bool(int(qaoa_row['classical_rank']) == 1), 'reported_exact_set_matches_reenumeration': exact_set_matches, 'exact_selected_tickers': ', '.join(exact_row['selected_tickers']), 'qaoa_selected_tickers': ', '.join(qaoa_row['selected_tickers'])}, name='value')
    return (table, summary)

def build_shortlist_opportunity_cost(*, validation_context: ClassicalValidationContext, base_solution: IndependentClassicalResult) -> pd.DataFrame:
    shortlist = validation_context.step6_handoff['shortlist']
    profile_results = validation_context.profiles
    records: list[dict[str, Any]] = []
    for profile_name in shortlist.index:
        profile = profile_results[profile_name]
        weights = _array(profile['result'].weights)
        components = objective_components_from_weights(validation_context=validation_context, policy_context=validation_context.base_context, weights=weights)
        economics = validation_context.step5.exact_goal_components(validation_context.base_context.portfolio_data, weights, validation_context.base_context.current_weights, validation_context.base_context.scenarios)
        audit = independent_constraint_audit(policy_context=validation_context.base_context, weights=weights, tolerance=validation_context.tolerances.feasibility)
        records.append({'profile': profile_name, 'base_policy_objective': float(components['total_objective']), 'objective_gap_to_base_global_optimum': max(float(components['total_objective'] - base_solution.objective_value), 0.0), 'expected_total_return': economics['expected_total_return'], 'volatility': economics['volatility'], 'worst_scenario_loss': economics['worst_scenario_loss'], 'gross_turnover': economics['gross_turnover'], 'total_trading_cost': economics['total_trading_cost'], 'hard_constraint_pass': bool(audit['satisfied'].all())})
    return pd.DataFrame(records).set_index('profile').sort_values('objective_gap_to_base_global_optimum')

def summarize_independent_result(result: IndependentClassicalResult) -> pd.Series:
    return pd.Series({'policy': result.policy, 'solver': result.solver, 'selected_solver_source': result.selected_solver_source, 'success': result.success, 'message': result.message, 'objective_value': result.objective_value, 'iterations': result.iterations, 'optimality': result.optimality, 'constraint_violation': result.constr_violation, 'kkt_stationarity_inf': result.kkt_stationarity_inf, 'kkt_complementarity_inf': result.kkt_complementarity_inf, 'dual_feasibility_min': result.dual_feasibility_min, 'solver_reported_duality_gap': result.solver_reported_duality_gap, 'kkt_certificate_pass': result.kkt_certificate_pass, 'hard_constraint_pass': bool(result.audit['satisfied'].all()), 'maximum_hard_violation': float(result.audit['absolute_violation'].max()), 'fixed_support_size': np.nan if result.fixed_active_tickers is None else len(result.fixed_active_tickers)}, name=result.label)

def build_validation_verdict(*, base_comparison: pd.Series, strict_comparison: pd.Series | None, qaoa_support_comparison: pd.Series, exact_support_comparison: pd.Series, qubo_summary: pd.Series, convexity: pd.Series) -> pd.Series:
    base_pass = base_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and bool(base_comparison['hard_constraint_pass']) and bool(base_comparison['independent_solver_success']) and bool(base_comparison['kkt_certificate_pass'])
    strict_pass = True
    if strict_comparison is not None:
        strict_pass = strict_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and bool(strict_comparison['hard_constraint_pass']) and bool(strict_comparison['independent_solver_success']) and bool(strict_comparison['kkt_certificate_pass'])
    support_pass = qaoa_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and exact_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH' and bool(qaoa_support_comparison['independent_solver_success']) and bool(exact_support_comparison['independent_solver_success']) and bool(qaoa_support_comparison['kkt_certificate_pass']) and bool(exact_support_comparison['kkt_certificate_pass'])
    convex = bool(convexity['continuous_problem_convex'])
    exact_reproduced = bool(qubo_summary['reported_exact_set_matches_reenumeration'])
    all_core = bool(base_pass and strict_pass and support_pass and convex and exact_reproduced)
    qaoa_exact = bool(qubo_summary['qaoa_exact_optimum'])
    if not all_core:
        verdict = 'FAIL_CLASSICAL_VALIDATION'
    elif qaoa_exact:
        verdict = 'PASS_QAOA_EXACT'
    else:
        verdict = 'PASS_QAOA_FEASIBLE_NEAR_OPTIMAL'
    return pd.Series({'continuous_problem_convex': convex, 'base_classical_global_optimum_reproduced': base_pass, 'strict_policy_optimum_reproduced': strict_pass, 'qaoa_fixed_support_refinement_reproduced': qaoa_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH', 'exact_fixed_support_refinement_reproduced': exact_support_comparison['verdict'] == 'PASS_OPTIMUM_MATCH', 'reported_exact_qubo_reproduced': exact_reproduced, 'independent_cvxpy_kkt_certificates_pass': bool(base_comparison['kkt_certificate_pass'] and (True if strict_comparison is None else strict_comparison['kkt_certificate_pass']) and qaoa_support_comparison['kkt_certificate_pass'] and exact_support_comparison['kkt_certificate_pass']), 'stored_starting_point_selectable': False, 'qaoa_exact_qubo_optimum': qaoa_exact, 'qaoa_qubo_rank': int(qubo_summary['qaoa_classical_rank']), 'qaoa_qubo_energy_gap': float(qubo_summary['qaoa_energy_gap']), 'overall_validation_verdict': verdict, 'supports_quantum_advantage_claim': False, 'supports_feasible_near_optimal_qaoa_claim': bool(all_core)}, name='value')


Writing step_07_validation_final.py


In [66]:
import sys
MODULE_NAME = 'step_07_validation_final'
sys.modules.pop(MODULE_NAME, None)
importlib.invalidate_caches()
import step_07_validation_final as step7
step7 = importlib.reload(step7)
required_functions = ['ValidationTolerances', 'ClassicalValidationContext', 'build_independent_problem', 'solve_independent_classical', 'cvxpy_solver_availability', 'convexity_certificate', 'independent_constraint_audit', 'objective_components_from_weights', 'compare_profile_to_solution', 'enumerate_qubo_exactly', 'build_shortlist_opportunity_cost', 'build_validation_verdict']
missing_functions = [name for name in required_functions if not hasattr(step7, name)]
if missing_functions:
    raise ImportError('The Step 7 module is incomplete: ' + ', '.join(missing_functions))
module_path = Path(step7.__file__).resolve()
module_source = module_path.read_text(encoding='utf-8')
required_markers = ['("CLARABEL", "OSQP")', 'independent_convex_qp_validator', 'stored starting point was initialization only', '_split_scipy_linear_constraints', 'kkt_certificate_pass', 'enumerate_qubo_exactly', 'PASS_QAOA_FEASIBLE_NEAR_OPTIMAL']
lower_module_source = module_source.lower()
missing_markers = [marker for marker in required_markers if marker.lower() not in lower_module_source]
if missing_markers:
    raise RuntimeError('Missing Step 7 final-validation markers: ' + ', '.join(missing_markers))
if 'candidate_x.append(x0)' in module_source:
    raise RuntimeError('The stored starting point is still selectable.')
print('PASS: Step 7 release module integrity check.')
print('Module:', module_path)


PASS: Step 7 module integrity check.
Module: /content/step_07_validation_final.py


## Step 7A — Declare the independent validation contract

In [67]:
STEP7_TOLERANCES = step7.ValidationTolerances(feasibility=5e-06, objective_match=2e-05, fixed_support_objective_match=3e-05, solver_weight_l1=0.02, qaoa_gap_zero=1e-08, hessian_psd=1e-09, kkt_stationarity=0.0001, kkt_complementarity=1e-05, dual_feasibility=1e-07)
STEP7_VALIDATION_CONTEXT = step7.ClassicalValidationContext(step4=step4, step5=step5, hybrid=hybrid, base_context=STEP5_CONTEXT, strict_context=STRICT_WARNING_CONTEXT, preferences=HYBRID_PREFERENCES, scales=GOAL_SCALES, mix=step5.GOAL_MIX, profiles=STEP5_PROFILE_RESULTS, step6_handoff=STEP6_STEP7_HANDOFF, tolerances=STEP7_TOLERANCES)
STEP7_VALIDATION_CONTEXT.validate()
STEP7_CVXPY_SOLVER_AVAILABILITY = step7.cvxpy_solver_availability()
display(STEP7_CVXPY_SOLVER_AVAILABILITY.to_frame())
if not bool(STEP7_CVXPY_SOLVER_AVAILABILITY['cvxpy_available']):
    raise RuntimeError('CVXPY is unavailable after the dependency cell.')
if not bool(STEP7_CVXPY_SOLVER_AVAILABILITY['clarabel_available'] or STEP7_CVXPY_SOLVER_AVAILABILITY['osqp_available']):
    raise RuntimeError('Step 7 requires Clarabel or OSQP through CVXPY.')
STEP7_VALIDATION_METHOD = pd.Series({'original_continuous_solver': 'Step 4 SciPy SLSQP', 'independent_reported_solver': 'CVXPY convex QP using Clarabel/OSQP', 'algorithmic_cross_check': 'SciPy trust-constr', 'secondary_diagnostic': 'SciPy SLSQP with separated equality and inequality constraints', 'stored_starting_point_role': 'initialization only; never selectable', 'binary_validator': 'complete fixed-cardinality enumeration', 'kkt_certificate': 'primal, stationarity, complementarity, and dual feasibility', 'full_space_policy': 'base soft-warning policy', 'strict_policy_validation': STRICT_WARNING_CONTEXT is not None, 'feasibility_tolerance': STEP7_TOLERANCES.feasibility, 'objective_match_tolerance': STEP7_TOLERANCES.objective_match, 'fixed_support_objective_tolerance': STEP7_TOLERANCES.fixed_support_objective_match, 'kkt_stationarity_tolerance': STEP7_TOLERANCES.kkt_stationarity, 'kkt_complementarity_tolerance': STEP7_TOLERANCES.kkt_complementarity, 'dual_feasibility_tolerance': STEP7_TOLERANCES.dual_feasibility}, name='value')
display(STEP7_VALIDATION_METHOD.to_frame())


,value
cvxpy_available,True
cvxpy_version,1.6.7
installed_solvers,"CLARABEL, CVXOPT, GLPK, GLPK_MI, HIGHS, OSQP, ..."
clarabel_available,True
osqp_available,True
error,


,value
original_continuous_solver,Step 4 SciPy SLSQP
independent_reported_solver,CVXPY convex QP using Clarabel/OSQP
algorithmic_cross_check,SciPy trust-constr
secondary_diagnostic,SciPy SLSQP with separated equality and inequa...
stored_starting_point_role,initialization only; never selectable
binary_validator,complete fixed-cardinality enumeration
kkt_certificate,"primal, stationarity, complementarity, and dua..."
full_space_policy,base soft-warning policy
strict_policy_validation,True
feasibility_tolerance,0.000005


## Step 7B — Convexity certificate

In [68]:
STEP7_CONVEXITY_CERTIFICATE = step7.convexity_certificate(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT)
display(STEP7_CONVEXITY_CERTIFICATE.to_frame())
if not bool(STEP7_CONVEXITY_CERTIFICATE['continuous_problem_convex']):
    raise RuntimeError('The Step 7 continuous problem failed the convexity certificate.')
print('PASS: the continuous validation problem is convex; a feasible optimum can be interpreted globally within numerical tolerance.')


,value
covariance_minimum_eigenvalue,0.000009
impact_minimum_eigenvalue,0.000199
objective_hessian_minimum_eigenvalue,0.0
penalty_coefficients_nonnegative,True
linear_constraint_system,True
continuous_problem_convex,True
global_optimum_interpretation,True


PASS: the continuous validation problem is convex; a feasible optimum can be interpreted globally within numerical tolerance.


## Step 7C — Exact classical validation of the binary QUBO

In [69]:
STEP7_EXACT_QUBO_ENUMERATION, STEP7_QUBO_VALIDATION_SUMMARY = step7.enumerate_qubo_exactly(model=HYBRID_RESULT['selection_model'], qaoa_selected_tickers=HYBRID_QAOA_PROFILE_RESULT['selected_tickers'], exact_selected_tickers=EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])
expected_state_count = math.comb(HYBRID_RESULT['selection_model'].n_variables, HYBRID_RESULT['selection_model'].cardinality)
assert len(STEP7_EXACT_QUBO_ENUMERATION) == expected_state_count
assert bool(STEP7_QUBO_VALIDATION_SUMMARY['reported_exact_set_matches_reenumeration'])
display(STEP7_QUBO_VALIDATION_SUMMARY.to_frame())
display(STEP7_EXACT_QUBO_ENUMERATION.head(15).style.format({'energy': '{:.8f}', 'classical_rank': '{:.0f}'}))
print('PASS: exact classical enumeration reproduced the reported reduced-QUBO optimum.')


,value
state_count,462
n_variables,11
cardinality,6
exact_energy,-4.067965
qaoa_energy,-4.067965
qaoa_energy_gap,0.0
qaoa_classical_rank,1
qaoa_exact_optimum,True
reported_exact_set_matches_reenumeration,True
exact_selected_tickers,"SGOV, BIL, SHY, QQQ, VUG, MTUM"


,bitstring,energy,selected_tickers,classical_rank
0,11111100000,-4.06796504,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'MTUM')",1
1,11111010000,-4.03567114,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'SPY')",2
2,11111000001,-3.99180645,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'USMV')",3
3,11111001000,-3.98432797,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'EWC')",4
4,11111000010,-3.97923632,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'VGK')",5
5,11110110000,-3.94426450,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'SPY')",6
6,11111000100,-3.92391106,"('SGOV', 'BIL', 'SHY', 'QQQ', 'VUG', 'EEM')",7
7,11110101000,-3.89534454,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'EWC')",8
8,11110100001,-3.89060267,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'USMV')",9
9,11110100010,-3.88426441,"('SGOV', 'BIL', 'SHY', 'QQQ', 'MTUM', 'VGK')",10


PASS: exact classical enumeration reproduced the reported reduced-QUBO optimum.


## Step 7D — Independently solve the full continuous problems

In [70]:
STEP7_BASE_CLASSICAL_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, label='Independent classical base-policy optimum', policy='base_soft_warning', start_weights=STEP5_PROFILE_RESULTS['Primary unrestricted classical']['result'].weights)
STEP7_STRICT_CLASSICAL_SOLUTION = None
if STRICT_WARNING_CONTEXT is not None and 'Classical strict-warning reference' in STEP5_PROFILE_RESULTS:
    STEP7_STRICT_CLASSICAL_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STRICT_WARNING_CONTEXT, label='Independent classical strict-warning optimum', policy='strict_warning', start_weights=STEP5_PROFILE_RESULTS['Classical strict-warning reference']['result'].weights)
solver_summaries = [step7.summarize_independent_result(STEP7_BASE_CLASSICAL_SOLUTION)]
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    solver_summaries.append(step7.summarize_independent_result(STEP7_STRICT_CLASSICAL_SOLUTION))
STEP7_INDEPENDENT_SOLVER_SUMMARY = pd.DataFrame(solver_summaries)
diagnostic_frames = []
for validation_name, solution in {'base_global': STEP7_BASE_CLASSICAL_SOLUTION, 'strict_global': STEP7_STRICT_CLASSICAL_SOLUTION}.items():
    if solution is None:
        continue
    frame = solution.solver_diagnostics.copy()
    frame.insert(0, 'validation_solution', validation_name)
    diagnostic_frames.append(frame)
STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS = pd.concat(diagnostic_frames, ignore_index=True)
display(STEP7_INDEPENDENT_SOLVER_SUMMARY.style.format({'objective_value': '{:.10f}', 'iterations': '{:.0f}', 'optimality': '{:.3e}', 'constraint_violation': '{:.3e}', 'kkt_stationarity_inf': '{:.3e}', 'kkt_complementarity_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}', 'solver_reported_duality_gap': '{:.3e}', 'maximum_hard_violation': '{:.3e}', 'fixed_support_size': '{:.0f}'}))
display(STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS.style.format({'exact_economic_objective': '{:.10f}', 'primal_residual_inf': '{:.3e}', 'stationarity_residual_inf': '{:.3e}', 'complementarity_residual_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}', 'solver_reported_duality_gap': '{:.3e}', 'iterations': '{:.0f}', 'solve_time_seconds': '{:.4f}'}))
for solution in [STEP7_BASE_CLASSICAL_SOLUTION, STEP7_STRICT_CLASSICAL_SOLUTION]:
    if solution is None:
        continue
    if not solution.selected_solver_source.startswith('cvxpy_'):
        raise RuntimeError('A non-CVXPY diagnostic result was selected as the independent solution.')
    if not bool(solution.kkt_certificate_pass):
        raise RuntimeError('The selected independent solution failed the KKT certificate.')
    if not bool(solution.success):
        raise RuntimeError(f'Independent optimization failed: {solution.label}')
print('PASS: full-space validation solutions came from independent CVXPY solvers and passed KKT checks.')


,policy,solver,selected_solver_source,success,message,objective_value,iterations,optimality,constraint_violation,kkt_stationarity_inf,kkt_complementarity_inf,dual_feasibility_min,solver_reported_duality_gap,kkt_certificate_pass,hard_constraint_pass,maximum_hard_violation,fixed_support_size
Independent classical base-policy optimum,base_soft_warning,independent CVXPY convex QP (Clarabel/OSQP),cvxpy_clarabel,True,selected independent solver=cvxpy_clarabel; trust-constr=`gtol` termination condition is satisfied.; split-constraint SLSQP=Optimization terminated successfully; stored starting point was initialization only and was not selectable,-0.7997879015,16,4.167e-15,4.372e-16,4.167e-15,2.964e-13,0.000e+00,nan,True,True,2.220e-16,nan
Independent classical strict-warning optimum,strict_warning,independent CVXPY convex QP (Clarabel/OSQP),cvxpy_clarabel,True,selected independent solver=cvxpy_clarabel; trust-constr=`gtol` termination condition is satisfied.; split-constraint SLSQP=Optimization terminated successfully; stored starting point was initialization only and was not selectable,-0.7826843669,20,4.643e-13,2.205e-15,4.643e-13,3.966e-12,0.000e+00,nan,True,True,7.772e-16,nan


,validation_solution,solver_source,role,status,finite_solution,feasible,exact_economic_objective,primal_residual_inf,stationarity_residual_inf,complementarity_residual_inf,dual_feasibility_min,solver_reported_duality_gap,kkt_certificate_pass,iterations,solve_time_seconds,selected,error
0,base_global,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7996195074,1.041e-17,2.353e-10,nan,nan,nan,False,37,nan,False,
1,base_global,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7997879015,1.110e-16,nan,nan,nan,nan,False,4,nan,False,
2,base_global,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7997879015,4.372e-16,4.167e-15,2.964e-13,0.000e+00,nan,True,16,0.0163,True,
3,base_global,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7997878917,9.949e-10,1.258e-09,7.671e-10,-7.144e-17,-1.800e-09,True,2225,0.0336,False,
4,strict_global,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7826518636,8.461e-13,3.316e-10,nan,nan,nan,False,40,nan,False,
5,strict_global,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7826843669,8.882e-16,nan,nan,nan,nan,False,12,nan,False,
6,strict_global,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7826843669,2.205e-15,4.643e-13,3.966e-12,0.000e+00,nan,True,20,0.0170,True,
7,strict_global,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7826843627,1.150e-09,7.359e-10,9.019e-10,-9.425e-17,-1.566e-09,True,1850,0.0529,False,


PASS: full-space validation solutions came from independent CVXPY solvers and passed KKT checks.


## Step 7E — Reproduce the stored full-space classical portfolios

In [71]:
STEP7_BASE_PROFILE_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, profile_name='Primary unrestricted classical', profile=STEP5_PROFILE_RESULTS['Primary unrestricted classical'], independent_result=STEP7_BASE_CLASSICAL_SOLUTION, validation_target='full-space base-policy global optimum')
STEP7_STRICT_PROFILE_VALIDATION = None
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    STEP7_STRICT_PROFILE_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STRICT_WARNING_CONTEXT, profile_name='Classical strict-warning reference', profile=STEP5_PROFILE_RESULTS['Classical strict-warning reference'], independent_result=STEP7_STRICT_CLASSICAL_SOLUTION, validation_target='full-space strict-warning global optimum')
profile_validation_rows = [STEP7_BASE_PROFILE_VALIDATION]
if STEP7_STRICT_PROFILE_VALIDATION is not None:
    profile_validation_rows.append(STEP7_STRICT_PROFILE_VALIDATION)
STEP7_FULL_SPACE_VALIDATION = pd.DataFrame(profile_validation_rows).set_index('profile')
display(STEP7_FULL_SPACE_VALIDATION.style.format({'stored_objective': '{:.10f}', 'independent_objective': '{:.10f}', 'signed_objective_gap': '{:.3e}', 'nonnegative_objective_gap': '{:.3e}', 'objective_match_tolerance': '{:.3e}', 'weight_l1_difference': '{:.3e}', 'weight_linf_difference': '{:.3e}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'kkt_stationarity_inf': '{:.3e}', 'kkt_complementarity_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}'}))
assert STEP7_FULL_SPACE_VALIDATION['independent_solver_success'].all()
assert STEP7_FULL_SPACE_VALIDATION['kkt_certificate_pass'].all()
assert STEP7_FULL_SPACE_VALIDATION['selected_solver_source'].str.startswith('cvxpy_').all()
print('PASS: stored full-space portfolios match independent CVXPY KKT-certified optima.')


,validation_target,policy,stored_objective,independent_objective,signed_objective_gap,nonnegative_objective_gap,objective_match_tolerance,weight_l1_difference,weight_linf_difference,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,hard_constraint_pass,independent_solver_success,selected_solver_source,kkt_certificate_pass,kkt_stationarity_inf,kkt_complementarity_inf,dual_feasibility_min,verdict
profile,,,,,,,,,,,,,,,,,,,,,,
Primary unrestricted classical,full-space base-policy global optimum,base_soft_warning,-0.7997879015,-0.7997879015,-1.998e-13,0.000e+00,2.000e-05,2.067e-10,1.025e-10,5.49%,7.86%,12.71%,13.09%,0.0020%,True,True,cvxpy_clarabel,True,4.167e-15,2.964e-13,0.000e+00,PASS_OPTIMUM_MATCH
Classical strict-warning reference,full-space strict-warning global optimum,strict_warning,-0.7826843669,-0.7826843669,-3.502e-12,0.000e+00,2.000e-05,9.076e-07,3.268e-07,5.48%,7.51%,12.17%,23.88%,0.0056%,True,True,cvxpy_clarabel,True,4.643e-13,3.966e-12,0.000e+00,PASS_OPTIMUM_MATCH


PASS: stored full-space portfolios match independent CVXPY KKT-certified optima.


## Step 7F — Validate continuous refinement on fixed active supports

In [72]:
STEP7_QAOA_ACTIVE_TICKERS = tuple(HYBRID_QAOA_PROFILE_RESULT['selected_tickers'])
STEP7_EXACT_ACTIVE_TICKERS = tuple(EXACT_ACTIVE_PROFILE_RESULT['selected_tickers'])
STEP7_QAOA_SUPPORT_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, label='Independent QAOA-support optimum', policy='base_soft_warning', start_weights=HYBRID_QAOA_PROFILE_RESULT['result'].weights, fixed_active_tickers=STEP7_QAOA_ACTIVE_TICKERS)
STEP7_EXACT_SUPPORT_SOLUTION = step7.solve_independent_classical(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, label='Independent exact-support optimum', policy='base_soft_warning', start_weights=EXACT_ACTIVE_PROFILE_RESULT['result'].weights, fixed_active_tickers=STEP7_EXACT_ACTIVE_TICKERS)
STEP7_QAOA_SUPPORT_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, profile_name='Independent Qiskit QAOA', profile=HYBRID_QAOA_PROFILE_RESULT, independent_result=STEP7_QAOA_SUPPORT_SOLUTION, validation_target='fixed QAOA active support', objective_tolerance=STEP7_TOLERANCES.fixed_support_objective_match)
STEP7_EXACT_SUPPORT_VALIDATION = step7.compare_profile_to_solution(validation_context=STEP7_VALIDATION_CONTEXT, policy_context=STEP5_CONTEXT, profile_name='Independent exact active-set benchmark', profile=EXACT_ACTIVE_PROFILE_RESULT, independent_result=STEP7_EXACT_SUPPORT_SOLUTION, validation_target='fixed exact active support', objective_tolerance=STEP7_TOLERANCES.fixed_support_objective_match)
STEP7_SUPPORT_VALIDATION = pd.DataFrame([STEP7_QAOA_SUPPORT_VALIDATION, STEP7_EXACT_SUPPORT_VALIDATION]).set_index('profile')
display(STEP7_SUPPORT_VALIDATION.style.format({'stored_objective': '{:.10f}', 'independent_objective': '{:.10f}', 'signed_objective_gap': '{:.3e}', 'nonnegative_objective_gap': '{:.3e}', 'objective_match_tolerance': '{:.3e}', 'weight_l1_difference': '{:.3e}', 'weight_linf_difference': '{:.3e}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}', 'kkt_stationarity_inf': '{:.3e}', 'kkt_complementarity_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}'}))
support_diagnostic_frames = []
for validation_name, solution in {'qaoa_fixed_support': STEP7_QAOA_SUPPORT_SOLUTION, 'exact_fixed_support': STEP7_EXACT_SUPPORT_SOLUTION}.items():
    frame = solution.solver_diagnostics.copy()
    frame.insert(0, 'validation_solution', validation_name)
    support_diagnostic_frames.append(frame)
STEP7_SUPPORT_SOLVER_DIAGNOSTICS = pd.concat(support_diagnostic_frames, ignore_index=True)
display(STEP7_SUPPORT_SOLVER_DIAGNOSTICS.style.format({'exact_economic_objective': '{:.10f}', 'primal_residual_inf': '{:.3e}', 'stationarity_residual_inf': '{:.3e}', 'complementarity_residual_inf': '{:.3e}', 'dual_feasibility_min': '{:.3e}', 'solver_reported_duality_gap': '{:.3e}', 'iterations': '{:.0f}', 'solve_time_seconds': '{:.4f}'}))
assert STEP7_SUPPORT_VALIDATION['independent_solver_success'].all()
assert STEP7_SUPPORT_VALIDATION['kkt_certificate_pass'].all()
assert STEP7_SUPPORT_VALIDATION['selected_solver_source'].str.startswith('cvxpy_').all()
print('PASS: QAOA and exact supports were independently reoptimized by CVXPY and passed KKT checks.')


,validation_target,policy,stored_objective,independent_objective,signed_objective_gap,nonnegative_objective_gap,objective_match_tolerance,weight_l1_difference,weight_linf_difference,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,hard_constraint_pass,independent_solver_success,selected_solver_source,kkt_certificate_pass,kkt_stationarity_inf,kkt_complementarity_inf,dual_feasibility_min,verdict
profile,,,,,,,,,,,,,,,,,,,,,,
Independent Qiskit QAOA,fixed QAOA active support,base_soft_warning,-0.7988178090,-0.7988178090,1.221e-15,1.221e-15,3.000e-05,1.573e-14,6.925e-15,5.45%,7.87%,12.72%,11.06%,0.0015%,True,True,cvxpy_osqp,True,7.772e-16,8.002e-18,0.000e+00,PASS_OPTIMUM_MATCH
Independent exact active-set benchmark,fixed exact active support,base_soft_warning,-0.7988178090,-0.7988178090,1.221e-15,1.221e-15,3.000e-05,1.573e-14,6.925e-15,5.45%,7.87%,12.72%,11.06%,0.0015%,True,True,cvxpy_osqp,True,7.772e-16,8.002e-18,0.000e+00,PASS_OPTIMUM_MATCH


,validation_solution,solver_source,role,status,finite_solution,feasible,exact_economic_objective,primal_residual_inf,stationarity_residual_inf,complementarity_residual_inf,dual_feasibility_min,solver_reported_duality_gap,kkt_certificate_pass,iterations,solve_time_seconds,selected,error
0,qaoa_fixed_support,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7988135636,2.202e-12,1.518e-10,nan,nan,nan,False,35,nan,False,
1,qaoa_fixed_support,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7988178090,1.332e-15,nan,nan,nan,nan,False,4,nan,False,
2,qaoa_fixed_support,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7988178090,4.125e-15,1.085e-13,5.751e-12,0.000e+00,nan,True,17,0.0138,False,
3,qaoa_fixed_support,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7988178090,1.110e-16,7.772e-16,8.002e-18,0.000e+00,2.033e-16,True,3100,0.0616,True,
4,exact_fixed_support,scipy_trust_constr,algorithmic_cross_check,`gtol` termination condition is satisfied.,True,True,-0.7988135636,2.202e-12,1.518e-10,nan,nan,nan,False,35,nan,False,
5,exact_fixed_support,scipy_slsqp_split_constraints,numerical_cross_check_only,Optimization terminated successfully,True,True,-0.7988178090,1.332e-15,nan,nan,nan,nan,False,4,nan,False,
6,exact_fixed_support,cvxpy_clarabel,independent_convex_qp_validator,optimal,True,True,-0.7988178090,4.125e-15,1.085e-13,5.751e-12,0.000e+00,nan,True,17,0.0123,False,
7,exact_fixed_support,cvxpy_osqp,independent_convex_qp_validator,optimal,True,True,-0.7988178090,1.110e-16,7.772e-16,8.002e-18,0.000e+00,2.033e-16,True,3100,0.0656,True,


PASS: QAOA and exact supports were independently reoptimized by CVXPY and passed KKT checks.


## Step 7G — Separate allocation correctness from selection quality

In [73]:
STEP7_SELECTION_VALIDATION = pd.Series({'qaoa_fixed_support_refinement_verdict': STEP7_QAOA_SUPPORT_VALIDATION['verdict'], 'exact_fixed_support_refinement_verdict': STEP7_EXACT_SUPPORT_VALIDATION['verdict'], 'qaoa_classical_qubo_rank': int(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_classical_rank']), 'qaoa_qubo_energy_gap': float(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_energy_gap']), 'qaoa_exact_qubo_optimum': bool(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_exact_optimum']), 'qaoa_financial_objective': float(STEP7_QAOA_SUPPORT_VALIDATION['stored_objective']), 'exact_active_financial_objective': float(STEP7_EXACT_SUPPORT_VALIDATION['stored_objective']), 'financial_objective_gap_qaoa_minus_exact': float(STEP7_QAOA_SUPPORT_VALIDATION['stored_objective'] - STEP7_EXACT_SUPPORT_VALIDATION['stored_objective']), 'qaoa_and_exact_support_overlap': float(len(set(STEP7_QAOA_ACTIVE_TICKERS) & set(STEP7_EXACT_ACTIVE_TICKERS)) / len(set(STEP7_QAOA_ACTIVE_TICKERS) | set(STEP7_EXACT_ACTIVE_TICKERS)))}, name='value')
display(STEP7_SELECTION_VALIDATION.to_frame())
print('Interpretation: a support-refinement PASS validates the classical allocation conditional on the support. The QUBO rank and energy gap validate the separate binary-selection quality.')


,value
qaoa_fixed_support_refinement_verdict,PASS_OPTIMUM_MATCH
exact_fixed_support_refinement_verdict,PASS_OPTIMUM_MATCH
qaoa_classical_qubo_rank,1
qaoa_qubo_energy_gap,0.0
qaoa_exact_qubo_optimum,True
qaoa_financial_objective,-0.798818
exact_active_financial_objective,-0.798818
financial_objective_gap_qaoa_minus_exact,0.0
qaoa_and_exact_support_overlap,1.0


Interpretation: a support-refinement PASS validates the classical allocation conditional on the support. The QUBO rank and energy gap validate the separate binary-selection quality.


## Step 7H — Opportunity cost of every Step 6 shortlist portfolio

In [74]:
STEP7_SHORTLIST_OPPORTUNITY_COST = step7.build_shortlist_opportunity_cost(validation_context=STEP7_VALIDATION_CONTEXT, base_solution=STEP7_BASE_CLASSICAL_SOLUTION)
display(STEP7_SHORTLIST_OPPORTUNITY_COST.style.format({'base_policy_objective': '{:.10f}', 'objective_gap_to_base_global_optimum': '{:.6f}', 'expected_total_return': '{:.2%}', 'volatility': '{:.2%}', 'worst_scenario_loss': '{:.2%}', 'gross_turnover': '{:.2%}', 'total_trading_cost': '{:.4%}'}))
assert STEP7_SHORTLIST_OPPORTUNITY_COST['hard_constraint_pass'].all()


,base_policy_objective,objective_gap_to_base_global_optimum,expected_total_return,volatility,worst_scenario_loss,gross_turnover,total_trading_cost,hard_constraint_pass
profile,,,,,,,,
Primary unrestricted classical,-0.7997879015,0.000000,5.49%,7.86%,12.71%,13.09%,0.0020%,True
Independent Qiskit QAOA,-0.7988178090,0.000970,5.45%,7.87%,12.72%,11.06%,0.0015%,True
Classical strict-warning reference,-0.7826843669,0.017104,5.48%,7.51%,12.17%,23.88%,0.0056%,True


## Step 7I — Independent hard-constraint residual audit

In [75]:
STEP7_SOLUTIONS = {'base_global': STEP7_BASE_CLASSICAL_SOLUTION, 'qaoa_fixed_support': STEP7_QAOA_SUPPORT_SOLUTION, 'exact_fixed_support': STEP7_EXACT_SUPPORT_SOLUTION}
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    STEP7_SOLUTIONS['strict_global'] = STEP7_STRICT_CLASSICAL_SOLUTION
audit_frames = []
audit_summary_records = []
for solution_name, solution in STEP7_SOLUTIONS.items():
    frame = solution.audit.copy()
    frame.insert(0, 'validation_solution', solution_name)
    audit_frames.append(frame)
    audit_summary_records.append({'validation_solution': solution_name, 'constraint_count': len(frame), 'failed_constraint_count': int((~frame['satisfied']).sum()), 'maximum_absolute_violation': float(frame['absolute_violation'].max()), 'solver_success': solution.success})
STEP7_ALL_CONSTRAINT_AUDITS = pd.concat(audit_frames, ignore_index=True)
STEP7_CONSTRAINT_AUDIT_SUMMARY = pd.DataFrame(audit_summary_records).set_index('validation_solution')
STEP7_FAILED_CONSTRAINTS = STEP7_ALL_CONSTRAINT_AUDITS.loc[~STEP7_ALL_CONSTRAINT_AUDITS['satisfied']].copy()
display(STEP7_CONSTRAINT_AUDIT_SUMMARY.style.format({'constraint_count': '{:.0f}', 'failed_constraint_count': '{:.0f}', 'maximum_absolute_violation': '{:.3e}'}))
if not STEP7_FAILED_CONSTRAINTS.empty:
    display(STEP7_FAILED_CONSTRAINTS)
    raise RuntimeError('A Step 7 independent validation solution failed a hard constraint.')
print('PASS: every independent classical validation solution satisfies every audited hard constraint.')


,constraint_count,failed_constraint_count,maximum_absolute_violation,solver_success
validation_solution,,,,
base_global,130,0,2.220e-16,True
qaoa_fixed_support,174,0,1.041e-17,True
exact_fixed_support,174,0,1.041e-17,True
strict_global,130,0,7.772e-16,True


PASS: every independent classical validation solution satisfies every audited hard constraint.


## Step 7J — Objective decomposition and reconciliation

In [76]:
objective_component_frames = []
for solution_name, solution in STEP7_SOLUTIONS.items():
    series = solution.objective_components.copy()
    frame = series.rename(solution_name).to_frame()
    objective_component_frames.append(frame)
STEP7_OBJECTIVE_COMPONENTS = pd.concat(objective_component_frames, axis=1)
STEP7_OBJECTIVE_RECONCILIATION = STEP7_OBJECTIVE_COMPONENTS.loc['total_objective'] - pd.Series({name: solution.objective_value for name, solution in STEP7_SOLUTIONS.items()})
display(STEP7_OBJECTIVE_COMPONENTS.style.format('{:.10f}'))
display(STEP7_OBJECTIVE_RECONCILIATION.rename('component_sum_minus_reported_objective').to_frame().style.format('{:.3e}'))
assert STEP7_OBJECTIVE_RECONCILIATION.abs().max() <= 1e-10
print('PASS: every independently reconstructed objective reconciles exactly to its component sum.')


,base_global,qaoa_fixed_support,exact_fixed_support,strict_global
growth_reward,-0.3541962876,-0.3538955984,-0.3538955984,-0.3445085184
income_reward,-0.5571580090,-0.5496884014,-0.5496884014,-0.5730628692
variance_penalty,0.0662464380,0.0663376940,0.0663376940,0.0603915039
concentration_penalty,0.0001142119,0.0001085649,0.0001085649,0.0001312405
linear_and_turnover_cost_penalty,0.0365686648,0.0298846316,0.0298846316,0.0694041672
market_impact_penalty,0.0012107456,0.0010434217,0.0010434217,0.0049601091
scenario_hinge_penalty,0.0074263348,0.0073918787,0.0073918787,0.0000000000
total_objective,-0.7997879015,-0.7988178090,-0.7988178090,-0.7826843669


,component_sum_minus_reported_objective
base_global,0.000e+00
qaoa_fixed_support,0.000e+00
exact_fixed_support,0.000e+00
strict_global,0.000e+00


PASS: every independently reconstructed objective reconciles exactly to its component sum.


## Step 7K — Weight-level solver agreement

In [77]:
weight_pairs = {'primary_classical': (STEP5_PROFILE_RESULTS['Primary unrestricted classical']['result'].weights, STEP7_BASE_CLASSICAL_SOLUTION.weights), 'qaoa_fixed_support': (HYBRID_QAOA_PROFILE_RESULT['result'].weights, STEP7_QAOA_SUPPORT_SOLUTION.weights), 'exact_fixed_support': (EXACT_ACTIVE_PROFILE_RESULT['result'].weights, STEP7_EXACT_SUPPORT_SOLUTION.weights)}
if STEP7_STRICT_CLASSICAL_SOLUTION is not None:
    weight_pairs['strict_warning'] = (STEP5_PROFILE_RESULTS['Classical strict-warning reference']['result'].weights, STEP7_STRICT_CLASSICAL_SOLUTION.weights)
weight_difference_records = []
weight_agreement_records = []
for comparison_name, (stored, independent) in weight_pairs.items():
    stored_series = pd.Series(np.asarray(stored, dtype=float), index=STEP5_CONTEXT.portfolio_data.tickers)
    independent_series = pd.Series(np.asarray(independent, dtype=float), index=STEP5_CONTEXT.portfolio_data.tickers)
    difference = stored_series - independent_series
    weight_agreement_records.append({'comparison': comparison_name, 'weight_l1_difference': float(difference.abs().sum()), 'weight_linf_difference': float(difference.abs().max())})
    for ticker, value in difference.abs().sort_values(ascending=False).head(10).items():
        weight_difference_records.append({'comparison': comparison_name, 'ticker': ticker, 'stored_weight': stored_series[ticker], 'independent_weight': independent_series[ticker], 'signed_difference': difference[ticker], 'absolute_difference': abs(value)})
STEP7_WEIGHT_AGREEMENT = pd.DataFrame(weight_agreement_records).set_index('comparison')
STEP7_TOP_WEIGHT_DIFFERENCES = pd.DataFrame(weight_difference_records)
display(STEP7_WEIGHT_AGREEMENT.style.format({'weight_l1_difference': '{:.3e}', 'weight_linf_difference': '{:.3e}'}))
display(STEP7_TOP_WEIGHT_DIFFERENCES.style.format({'stored_weight': '{:.4%}', 'independent_weight': '{:.4%}', 'signed_difference': '{:+.3e}', 'absolute_difference': '{:.3e}'}))


,weight_l1_difference,weight_linf_difference
comparison,,
primary_classical,2.067e-10,1.025e-10
qaoa_fixed_support,1.573e-14,6.925e-15
exact_fixed_support,1.573e-14,6.925e-15
strict_warning,9.076e-07,3.268e-07


,comparison,ticker,stored_weight,independent_weight,signed_difference,absolute_difference
0,primary_classical,SGOV,7.9749%,7.9749%,+1.025e-10,1.025e-10
1,primary_classical,UUP,0.0000%,0.0000%,-6.129e-11,6.129e-11
2,primary_classical,MTUM,1.1468%,1.1468%,-4.167e-11,4.167e-11
3,primary_classical,IEF,3.1895%,3.1895%,+3.515e-13,3.515e-13
4,primary_classical,XLE,0.9967%,0.9967%,-2.920e-13,2.920e-13
5,primary_classical,SPY,2.4840%,2.4840%,+2.433e-13,2.433e-13
6,primary_classical,EWC,1.8943%,1.8943%,+5.289e-14,5.289e-14
7,primary_classical,VGK,1.9762%,1.9762%,+2.382e-14,2.382e-14
8,primary_classical,USMV,3.4464%,3.4464%,+2.322e-14,2.322e-14
9,primary_classical,MUB,2.0415%,2.0415%,-1.834e-14,1.834e-14


## Step 7L — Classical validation figures

In [78]:
STEP7_OUTPUT = Path(OUTPUT_ROOT) / 'step7_release_final' / DATA_SOURCE / STEP4_COST_SCENARIO
STEP7_FIGURE_DIR = STEP7_OUTPUT / 'figures'
STEP7_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
objective_gap_plot = STEP7_SHORTLIST_OPPORTUNITY_COST['objective_gap_to_base_global_optimum'].sort_values(ascending=False)
x_positions = np.arange(len(objective_gap_plot))
fig = plt.figure(figsize=(11, 6))
plt.bar(x_positions, objective_gap_plot.to_numpy(dtype=float))
plt.xticks(x_positions, [label.replace('Independent ', '').replace('Classical ', '') for label in objective_gap_plot.index], rotation=45, ha='right')
plt.ylabel('Objective gap to independent base optimum')
plt.title('Step 7 release: Independent Convex-QP Opportunity Cost')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP7_FIGURE_DIR / 'shortlist_objective_gaps.png', dpi=180, bbox_inches='tight')
plt.show()


In [79]:
weight_gap_plot = STEP7_WEIGHT_AGREEMENT['weight_l1_difference'].sort_values(ascending=False)
x_positions = np.arange(len(weight_gap_plot))
fig = plt.figure(figsize=(10, 5))
plt.bar(x_positions, weight_gap_plot.to_numpy(dtype=float))
plt.xticks(x_positions, weight_gap_plot.index, rotation=35, ha='right')
plt.ylabel('L1 distance between stored and independent weights')
plt.title('Step 7: Weight-Level Solver Agreement')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP7_FIGURE_DIR / 'weight_solver_agreement.png', dpi=180, bbox_inches='tight')
plt.show()


In [80]:
qubo_plot = STEP7_EXACT_QUBO_ENUMERATION.head(15).copy()
x_positions = np.arange(len(qubo_plot))
fig = plt.figure(figsize=(11, 5))
plt.bar(x_positions, qubo_plot['energy'].to_numpy(dtype=float))
qaoa_rank = int(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_classical_rank'])
if qaoa_rank <= len(qubo_plot):
    plt.scatter([qaoa_rank - 1], [qubo_plot.iloc[qaoa_rank - 1]['energy']], marker='x', s=120, label='QAOA-selected state')
    plt.legend()
plt.xticks(x_positions, qubo_plot['classical_rank'].astype(int))
plt.xlabel('Exact classical rank')
plt.ylabel('Economic QUBO energy')
plt.title('Step 7: Exact Classical QUBO Ordering')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(STEP7_FIGURE_DIR / 'exact_qubo_ordering.png', dpi=180, bbox_inches='tight')
plt.show()


## Step 7M — Formal final-validation verdict

In [81]:
STEP7_SELECTED_SOLVER_SOURCES = pd.Series(
    {
        "base_global": (
            STEP7_BASE_CLASSICAL_SOLUTION
            .selected_solver_source
        ),
        "qaoa_fixed_support": (
            STEP7_QAOA_SUPPORT_SOLUTION
            .selected_solver_source
        ),
        "exact_fixed_support": (
            STEP7_EXACT_SUPPORT_SOLUTION
            .selected_solver_source
        ),
        **(
            {}
            if STEP7_STRICT_CLASSICAL_SOLUTION is None
            else {
                "strict_global": (
                    STEP7_STRICT_CLASSICAL_SOLUTION
                    .selected_solver_source
                )
            }
        ),
    },
    name="selected_solver_source",
)

STEP7_KKT_CERTIFICATES = pd.Series(
    {
        "base_global": (
            STEP7_BASE_CLASSICAL_SOLUTION
            .kkt_certificate_pass
        ),
        "qaoa_fixed_support": (
            STEP7_QAOA_SUPPORT_SOLUTION
            .kkt_certificate_pass
        ),
        "exact_fixed_support": (
            STEP7_EXACT_SUPPORT_SOLUTION
            .kkt_certificate_pass
        ),
        **(
            {}
            if STEP7_STRICT_CLASSICAL_SOLUTION is None
            else {
                "strict_global": (
                    STEP7_STRICT_CLASSICAL_SOLUTION
                    .kkt_certificate_pass
                )
            }
        ),
    },
    name="kkt_certificate_pass",
)

STEP7_INDEPENDENCE_CERTIFICATE = pd.Series(
    {
        "stored_starting_point_selectable": False,
        "all_selected_solvers_are_cvxpy": bool(
            STEP7_SELECTED_SOLVER_SOURCES
            .str.startswith("cvxpy_")
            .all()
        ),
        "all_kkt_certificates_pass": bool(
            STEP7_KKT_CERTIFICATES.all()
        ),
        "scipy_results_are_diagnostic_only": True,
        "independent_solver_requirement_pass": bool(
            STEP7_SELECTED_SOLVER_SOURCES
            .str.startswith("cvxpy_")
            .all()
            and STEP7_KKT_CERTIFICATES.all()
        ),
    },
    name="value",
)

display(
    STEP7_SELECTED_SOLVER_SOURCES.to_frame()
)
display(
    STEP7_KKT_CERTIFICATES.to_frame()
)
display(
    STEP7_INDEPENDENCE_CERTIFICATE.to_frame()
)

STEP7_VALIDATION_VERDICT = (
    step7.build_validation_verdict(
        base_comparison=(
            STEP7_BASE_PROFILE_VALIDATION
        ),
        strict_comparison=(
            STEP7_STRICT_PROFILE_VALIDATION
        ),
        qaoa_support_comparison=(
            STEP7_QAOA_SUPPORT_VALIDATION
        ),
        exact_support_comparison=(
            STEP7_EXACT_SUPPORT_VALIDATION
        ),
        qubo_summary=(
            STEP7_QUBO_VALIDATION_SUMMARY
        ),
        convexity=(
            STEP7_CONVEXITY_CERTIFICATE
        ),
    )
)

STEP7_DEVELOPMENT_VALIDATION_PASS = bool(
    STEP7_VALIDATION_VERDICT[
        "overall_validation_verdict"
    ]
    != "FAIL_CLASSICAL_VALIDATION"
    and STEP7_INDEPENDENCE_CERTIFICATE[
        "independent_solver_requirement_pass"
    ]
)

STEP7_FINAL_EVIDENCE_READY = bool(
    STEP7_DEVELOPMENT_VALIDATION_PASS
    and len(
        QAOA_SEED_SUMMARY
    ) >= 20
    and bool(
        STEP6_STEP7_HANDOFF[
            "final_forward_evidence_ready"
        ]
    )
    and not bool(FAST_MODE)
)

STEP7_EVIDENCE_TIER = (
    "FINAL_CHALLENGE_EVIDENCE"
    if STEP7_FINAL_EVIDENCE_READY
    else "DEVELOPMENT_VALIDATION"
)

STEP7_VALIDATION_VERDICT.loc[
    "development_validation_pass"
] = STEP7_DEVELOPMENT_VALIDATION_PASS
STEP7_VALIDATION_VERDICT.loc[
    "final_evidence_ready"
] = STEP7_FINAL_EVIDENCE_READY
STEP7_VALIDATION_VERDICT.loc[
    "evidence_tier"
] = STEP7_EVIDENCE_TIER
STEP7_VALIDATION_VERDICT.loc[
    "qaoa_seed_count"
] = len(
    QAOA_SEED_SUMMARY
)
STEP7_VALIDATION_VERDICT.loc[
    "stored_starting_point_selectable"
] = False
STEP7_VALIDATION_VERDICT.loc[
    "independent_solver_requirement_pass"
] = bool(
    STEP7_INDEPENDENCE_CERTIFICATE[
        "independent_solver_requirement_pass"
    ]
)

display(
    STEP7_VALIDATION_VERDICT.to_frame()
)

if not STEP7_DEVELOPMENT_VALIDATION_PASS:
    raise RuntimeError(
        "Step 7 final classical validation failed."
    )

print(
    "PASS: Step 7 final independent classical "
    "validation completed."
)
print(
    "Overall verdict:",
    STEP7_VALIDATION_VERDICT[
        "overall_validation_verdict"
    ],
)
print(
    "Evidence tier:",
    STEP7_EVIDENCE_TIER,
)
print(
    "Stored starting point selectable:",
    False,
)


,selected_solver_source
base_global,cvxpy_clarabel
qaoa_fixed_support,cvxpy_osqp
exact_fixed_support,cvxpy_osqp
strict_global,cvxpy_clarabel


,kkt_certificate_pass
base_global,True
qaoa_fixed_support,True
exact_fixed_support,True
strict_global,True


,value
stored_starting_point_selectable,False
all_selected_solvers_are_cvxpy,True
all_kkt_certificates_pass,True
scipy_results_are_diagnostic_only,True
independent_solver_requirement_pass,True


,value
continuous_problem_convex,True
base_classical_global_optimum_reproduced,True
strict_policy_optimum_reproduced,True
qaoa_fixed_support_refinement_reproduced,True
exact_fixed_support_refinement_reproduced,True
reported_exact_qubo_reproduced,True
independent_cvxpy_kkt_certificates_pass,True
stored_starting_point_selectable,False
qaoa_exact_qubo_optimum,True
qaoa_qubo_rank,1


PASS: Step 7 final independent classical validation completed.
Overall verdict: PASS_QAOA_EXACT
Evidence tier: FINAL_CHALLENGE_EVIDENCE
Stored starting point selectable: False


## Step 7 interpretation

## Step 7N — Handoff to Step 8

In [82]:
STEP7_STEP8_HANDOFF = {'step6_handoff': STEP6_STEP7_HANDOFF, 'validation_context': STEP7_VALIDATION_CONTEXT, 'validation_method': STEP7_VALIDATION_METHOD, 'cvxpy_solver_availability': STEP7_CVXPY_SOLVER_AVAILABILITY, 'selected_solver_sources': STEP7_SELECTED_SOLVER_SOURCES, 'kkt_certificates': STEP7_KKT_CERTIFICATES, 'independence_certificate': STEP7_INDEPENDENCE_CERTIFICATE, 'full_space_solver_diagnostics': STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS, 'support_solver_diagnostics': STEP7_SUPPORT_SOLVER_DIAGNOSTICS, 'convexity_certificate': STEP7_CONVEXITY_CERTIFICATE, 'exact_qubo_enumeration': STEP7_EXACT_QUBO_ENUMERATION, 'qubo_validation_summary': STEP7_QUBO_VALIDATION_SUMMARY, 'base_classical_solution': STEP7_BASE_CLASSICAL_SOLUTION, 'strict_classical_solution': STEP7_STRICT_CLASSICAL_SOLUTION, 'qaoa_support_solution': STEP7_QAOA_SUPPORT_SOLUTION, 'exact_support_solution': STEP7_EXACT_SUPPORT_SOLUTION, 'full_space_validation': STEP7_FULL_SPACE_VALIDATION, 'support_validation': STEP7_SUPPORT_VALIDATION, 'selection_validation': STEP7_SELECTION_VALIDATION, 'shortlist_opportunity_cost': STEP7_SHORTLIST_OPPORTUNITY_COST, 'constraint_audit_summary': STEP7_CONSTRAINT_AUDIT_SUMMARY, 'objective_components': STEP7_OBJECTIVE_COMPONENTS, 'weight_agreement': STEP7_WEIGHT_AGREEMENT, 'validation_verdict': STEP7_VALIDATION_VERDICT, 'development_validation_pass': STEP7_DEVELOPMENT_VALIDATION_PASS, 'final_evidence_ready': STEP7_FINAL_EVIDENCE_READY, 'evidence_tier': STEP7_EVIDENCE_TIER}
assert STEP7_STEP8_HANDOFF['development_validation_pass']
assert bool(STEP7_INDEPENDENCE_CERTIFICATE['independent_solver_requirement_pass'])
assert STEP7_SELECTED_SOLVER_SOURCES.str.startswith('cvxpy_').all()
assert STEP7_KKT_CERTIFICATES.all()
assert STEP7_FULL_SPACE_VALIDATION['hard_constraint_pass'].all()
assert STEP7_SUPPORT_VALIDATION['hard_constraint_pass'].all()
assert bool(STEP7_QUBO_VALIDATION_SUMMARY['reported_exact_set_matches_reenumeration'])
print('PASS: Step 8 handoff contains CVXPY-validated, KKT-certified classical and quantum-assisted portfolios.')
print('Step 7 verdict:', STEP7_VALIDATION_VERDICT['overall_validation_verdict'])
print('Final evidence ready:', STEP7_FINAL_EVIDENCE_READY)


PASS: Step 8 handoff contains CVXPY-validated, KKT-certified classical and quantum-assisted portfolios.
Step 7 verdict: PASS_QAOA_EXACT
Final evidence ready: True


## Step 7O — Export and download the complete Steps 3–7 package

In [83]:
STEP7_OUTPUT.mkdir(parents=True, exist_ok=True)
STEP7_VALIDATION_METHOD.to_csv(STEP7_OUTPUT / 'validation_method.csv')
STEP7_CVXPY_SOLVER_AVAILABILITY.to_csv(STEP7_OUTPUT / 'cvxpy_solver_availability.csv')
STEP7_SELECTED_SOLVER_SOURCES.to_csv(STEP7_OUTPUT / 'selected_solver_sources.csv')
STEP7_KKT_CERTIFICATES.to_csv(STEP7_OUTPUT / 'kkt_certificates.csv')
STEP7_INDEPENDENCE_CERTIFICATE.to_csv(STEP7_OUTPUT / 'independence_certificate.csv')
STEP7_FULL_SPACE_SOLVER_DIAGNOSTICS.to_csv(STEP7_OUTPUT / 'full_space_solver_diagnostics.csv', index=False)
STEP7_SUPPORT_SOLVER_DIAGNOSTICS.to_csv(STEP7_OUTPUT / 'support_solver_diagnostics.csv', index=False)
STEP7_CONVEXITY_CERTIFICATE.to_csv(STEP7_OUTPUT / 'convexity_certificate.csv')
STEP7_EXACT_QUBO_ENUMERATION.to_csv(STEP7_OUTPUT / 'exact_qubo_enumeration.csv', index=False)
STEP7_QUBO_VALIDATION_SUMMARY.to_csv(STEP7_OUTPUT / 'qubo_validation_summary.csv')
STEP7_INDEPENDENT_SOLVER_SUMMARY.to_csv(STEP7_OUTPUT / 'independent_solver_summary.csv')
STEP7_FULL_SPACE_VALIDATION.to_csv(STEP7_OUTPUT / 'full_space_validation.csv')
STEP7_SUPPORT_VALIDATION.to_csv(STEP7_OUTPUT / 'fixed_support_validation.csv')
STEP7_SELECTION_VALIDATION.to_csv(STEP7_OUTPUT / 'selection_validation.csv')
STEP7_SHORTLIST_OPPORTUNITY_COST.to_csv(STEP7_OUTPUT / 'shortlist_opportunity_cost.csv')
STEP7_CONSTRAINT_AUDIT_SUMMARY.to_csv(STEP7_OUTPUT / 'constraint_audit_summary.csv')
STEP7_ALL_CONSTRAINT_AUDITS.to_csv(STEP7_OUTPUT / 'all_independent_constraint_audits.csv', index=False)
STEP7_OBJECTIVE_COMPONENTS.to_csv(STEP7_OUTPUT / 'objective_components.csv')
STEP7_WEIGHT_AGREEMENT.to_csv(STEP7_OUTPUT / 'weight_agreement.csv')
STEP7_TOP_WEIGHT_DIFFERENCES.to_csv(STEP7_OUTPUT / 'top_weight_differences.csv', index=False)
STEP7_VALIDATION_VERDICT.to_csv(STEP7_OUTPUT / 'validation_verdict.csv')
validated_weights = pd.DataFrame({name: solution.weights for name, solution in STEP7_SOLUTIONS.items()})
validated_weights.to_csv(STEP7_OUTPUT / 'independent_validated_weights.csv')
step7_metadata = {'method_version': 'step_07_validation_final_final', 'data_source': DATA_SOURCE, 'cost_scenario': STEP4_COST_SCENARIO, 'risk_policy_mode': RISK_POLICY_MODE, 'continuous_problem_convex': bool(STEP7_CONVEXITY_CERTIFICATE['continuous_problem_convex']), 'qaoa_classical_rank': int(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_classical_rank']), 'qaoa_qubo_energy_gap': float(STEP7_QUBO_VALIDATION_SUMMARY['qaoa_energy_gap']), 'overall_validation_verdict': str(STEP7_VALIDATION_VERDICT['overall_validation_verdict']), 'development_validation_pass': bool(STEP7_DEVELOPMENT_VALIDATION_PASS), 'final_evidence_ready': bool(STEP7_FINAL_EVIDENCE_READY), 'evidence_tier': STEP7_EVIDENCE_TIER, 'qaoa_seed_count': int(len(QAOA_SEED_SUMMARY)), 'forward_evidence_ready': bool(STEP6_STEP7_HANDOFF['final_forward_evidence_ready']), 'selected_solver_sources': STEP7_SELECTED_SOLVER_SOURCES.to_dict(), 'stored_starting_point_selectable': False, 'independent_solver_requirement_pass': bool(STEP7_INDEPENDENCE_CERTIFICATE['independent_solver_requirement_pass']), 'all_kkt_certificates_pass': bool(STEP7_KKT_CERTIFICATES.all()), 'supports_quantum_advantage_claim': False, 'synthetic_results_are_not_historical_backtests': DATA_SOURCE == 'synthetic'}
(STEP7_OUTPUT / 'step7_metadata.json').write_text(json.dumps(step7_metadata, indent=2), encoding='utf-8')
required_exports = [STEP7_OUTPUT / 'convexity_certificate.csv', STEP7_OUTPUT / 'independence_certificate.csv', STEP7_OUTPUT / 'full_space_solver_diagnostics.csv', STEP7_OUTPUT / 'support_solver_diagnostics.csv', STEP7_OUTPUT / 'exact_qubo_enumeration.csv', STEP7_OUTPUT / 'full_space_validation.csv', STEP7_OUTPUT / 'fixed_support_validation.csv', STEP7_OUTPUT / 'validation_verdict.csv', STEP7_OUTPUT / 'step7_metadata.json']
missing_exports = [str(path) for path in required_exports if not path.exists()]
if missing_exports:
    raise FileNotFoundError('Step 7 export is incomplete: ' + ', '.join(missing_exports))
archive_base = Path('/content') / f'step3_step4_step5q_step6_step7_finance_release_final_{DATA_SOURCE}_{STEP4_COST_SCENARIO}'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=Path(OUTPUT_ROOT))
print('Created complete Steps 3–7 package:', archive_path)
print('Archive size:', f'{Path(archive_path).stat().st_size / 1024 ** 2:.2f} MB')
files.download(archive_path)


Created complete Steps 3–7 package: /content/portfolio_pipeline_steps_03_to_07_synthetic_base.zip
Archive size: 2.59 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Step 7 final validation discipline